In [1]:

from pathlib import Path
import pandas as pd
import ast
import time

DATA_FILE = Path("data") / "clean_training_data.csv"

start = time.time()
df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nLabel counts:")
print(df["accepted"].value_counts())
print(f"\nLoading time: {time.time() - start:.2f} seconds")


Dataset loaded successfully!
Shape: (787138, 13)

Label counts:
accepted
0    498290
1    288848
Name: count, dtype: int64

Loading time: 2.71 seconds


In [2]:

feature_columns = [
    "form_a", "form_b", "form_c", "form_f", "form_g",
    "form_h", "form_i", "form_j", "form_k"
]

for column in feature_columns:
    print(f"\n{column}")
    print(df[column].value_counts().head(5))



form_a
form_a
['1']                   502230
['0', '1', '2', '3']    128413
['2']                    69856
['0', '1']               28743
['0', '2']               15340
Name: count, dtype: int64

form_b
form_b
['2']                   270763
['1']                   235318
['0', '1', '2', '3']    127945
['0']                    87019
['3']                    66073
Name: count, dtype: int64

form_c
form_c
['2']      383111
['1']      168801
['3']      157123
['0']       78073
missing        30
Name: count, dtype: int64

form_f
form_f
['3', '5', '7']                   30260
['2', '3', '5', '7']              26729
['0', '1', '2', '3', '5', '7']    26419
['1', '2', '3', '5', '7']         25992
['2']                             20093
Name: count, dtype: int64

form_g
form_g
['0']                        48606
['2']                        29596
['0', '1']                   27542
['0', '2']                   24928
['0', '1', '2', '3', '4']    22181
Name: count, dtype: int64

form_h
form_h
['1']

In [3]:

import re
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

print("Starting feature encoding...")

feature_columns = [
    "form_a", "form_b", "form_c", "form_f", "form_g",
    "form_h", "form_i", "form_j", "form_k"
]

sparse_features = []
encoders = {}

# Encode questionnaire selections as separate binary features
for column in feature_columns:
    print(f"Encoding {column}...")

    values = df[column].fillna("missing").astype(str)

    selections = [
        re.findall(r"\d+", value)
        if value != "missing" else ["missing"]
        for value in values
    ]

    # Preserve an explicit missing token if present
    selections = [
        selection if selection else ["missing"]
        for selection in selections
    ]

    encoder = MultiLabelBinarizer(sparse_output=True)
    encoded = encoder.fit_transform(selections)

    encoders[column] = encoder
    sparse_features.append(encoded)

    print(f"  Created {encoded.shape[1]} features")

# Encode destination identity as a categorical feature
print("Encoding destination IDs...")

destination_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

destination_features = destination_encoder.fit_transform(
    df[["destination_id"]]
)

sparse_features.append(destination_features)

# Combine sparse features without creating a large dense matrix
X = hstack(sparse_features, format="csr", dtype=np.float32)
y = df["accepted"].to_numpy(dtype=np.int8)
groups = df["traveler_id"].to_numpy()

print("\nEncoding completed!")
print("Feature matrix shape:", X.shape)
print("Feature matrix format:", X.format)
print("Labels shape:", y.shape)
print("Sparse matrix storage:", round(X.data.nbytes / (1024**2), 2), "MB")


Starting feature encoding...
Encoding form_a...
  Created 5 features
Encoding form_b...
  Created 5 features
Encoding form_c...
  Created 5 features
Encoding form_f...
  Created 9 features
Encoding form_g...
  Created 9 features
Encoding form_h...
  Created 4 features
Encoding form_i...
  Created 4 features
Encoding form_j...
  Created 4 features
Encoding form_k...
  Created 4 features
Encoding destination IDs...

Encoding completed!
Feature matrix shape: (787138, 1501)
Feature matrix format: csr
Labels shape: (787138,)
Sparse matrix storage: 49.3 MB


In [4]:

from sklearn.model_selection import GroupShuffleSplit
import numpy as np

print("Splitting data by traveler...")

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

train_travelers = set(groups[train_idx])
test_travelers = set(groups[test_idx])

print("\nSplit completed!")
print("Training examples:", len(train_idx))
print("Testing examples:", len(test_idx))
print("Training travelers:", len(train_travelers))
print("Testing travelers:", len(test_travelers))
print("Travelers in both splits:", len(train_travelers & test_travelers))

print("\nTraining label distribution:")
print("Rejected (0):", int(np.sum(y_train == 0)))
print("Accepted (1):", int(np.sum(y_train == 1)))

print("\nTesting label distribution:")
print("Rejected (0):", int(np.sum(y_test == 0)))
print("Accepted (1):", int(np.sum(y_test == 1)))


Splitting data by traveler...

Split completed!
Training examples: 630586
Testing examples: 156552
Training travelers: 48708
Testing travelers: 12178
Travelers in both splits: 0

Training label distribution:
Rejected (0): 399170
Accepted (1): 231416

Testing label distribution:
Rejected (0): 99120
Accepted (1): 57432


In [5]:

import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Starting Logistic Regression training...")
start = time.time()

model = LogisticRegression(
    max_iter=200,
    solver="saga",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print(f"\nTraining completed in {time.time() - start:.2f} seconds.")

print("\nEvaluating model...")
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print("\n--- Model Evaluation ---")
print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print("Precision:", round(precision_score(y_test, predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, predictions, zero_division=0), 4))
print("F1 score:", round(f1_score(y_test, predictions, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, probabilities), 4))

print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=["Rejected", "Accepted"]))


Starting Logistic Regression training...


C:\Users\Chandrashekar\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



Training completed in 47.26 seconds.

Evaluating model...

--- Model Evaluation ---
Accuracy: 0.6528
Precision: 0.5195
Recall: 0.7119
F1 score: 0.6007
ROC-AUC: 0.7216

Classification Report:
              precision    recall  f1-score   support

    Rejected       0.79      0.62      0.69     99120
    Accepted       0.52      0.71      0.60     57432

    accuracy                           0.65    156552
   macro avg       0.65      0.67      0.65    156552
weighted avg       0.69      0.65      0.66    156552



In [6]:

from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(
    y_test, probabilities
)

print("Threshold    Precision    Recall")

for target in [0.50, 0.55, 0.60, 0.65, 0.70]:
    idx = np.argmin(np.abs(thresholds - target))

    print(
        f"{thresholds[idx]:.3f}        "
        f"{precision[idx]:.3f}        "
        f"{recall[idx]:.3f}"
    )


Threshold    Precision    Recall
0.500        0.520        0.712
0.550        0.547        0.627
0.600        0.578        0.524
0.650        0.615        0.399
0.700        0.651        0.263


In [7]:
from sklearn.metrics import classification_report, confusion_matrix

chosen_threshold = 0.55

threshold_predictions = (
    probabilities >= chosen_threshold
).astype(int)

print("Chosen threshold:", chosen_threshold)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, threshold_predictions))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        threshold_predictions,
        target_names=["Rejected", "Accepted"]
    )
)

Chosen threshold: 0.55

Confusion Matrix:
[[69263 29857]
 [21412 36020]]

Classification Report:
              precision    recall  f1-score   support

    Rejected       0.76      0.70      0.73     99120
    Accepted       0.55      0.63      0.58     57432

    accuracy                           0.67    156552
   macro avg       0.66      0.66      0.66    156552
weighted avg       0.68      0.67      0.68    156552



In [8]:
from pathlib import Path
import joblib
import time

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

artifacts = {
    "model": model,
    "encoders": encoders,
    "destination_encoder": destination_encoder,
    "feature_columns": feature_columns,
    "threshold": 0.55,
}

model_path = MODEL_DIR / "trip_recommendation_model.joblib"

joblib.dump(artifacts, model_path)

print("Model saved successfully!")
print("Location:", model_path.resolve())
print(f"File size: {model_path.stat().st_size / (1024 * 1024):.2f} MB")

Model saved successfully!
Location: C:\Users\Chandrashekar\OneDrive\Desktop\Place Recommandation\models\trip_recommendation_model.joblib
File size: 0.02 MB


In [9]:
import joblib
from pathlib import Path

model_path = Path("models") / "trip_recommendation_model.joblib"

saved_artifacts = joblib.load(model_path)

loaded_model = saved_artifacts["model"]
loaded_encoders = saved_artifacts["encoders"]
loaded_destination_encoder = saved_artifacts["destination_encoder"]
loaded_feature_columns = saved_artifacts["feature_columns"]
loaded_threshold = saved_artifacts["threshold"]

print("Saved model loaded successfully!")
print("Model type:", type(loaded_model).__name__)
print("Questionnaire features:", loaded_feature_columns)
print("Recommendation threshold:", loaded_threshold)


Saved model loaded successfully!
Model type: LogisticRegression
Questionnaire features: ['form_a', 'form_b', 'form_c', 'form_f', 'form_g', 'form_h', 'form_i', 'form_j', 'form_k']
Recommendation threshold: 0.55


In [10]:
from pathlib import Path

mapping_path = Path("data") / "destination_ids.txt"

with open(mapping_path, "r", encoding="utf-8") as file:
    destination_names = [line.strip() for line in file]

print("Destination mapping loaded successfully!")
print("Total destinations:", len(destination_names))

print("\nSample destinations:")
for destination_id in [836, 837, 0, 1, 2]:
    print(f"{destination_id}: {destination_names[destination_id]}")

Destination mapping loaded successfully!
Total destinations: 1486

Sample destinations:
836: Al Ain, United Arab Emirates
837: Dubai, United Arab Emirates
0: Butrint, Albania
1: Tirana, Albania
2: Algiers, Algeria


In [11]:
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

def recommend_destinations(traveler_answers, top_n=5):
    # Encode the traveler's questionnaire answers
    feature_matrices = []

    for column in loaded_feature_columns:
        answer = str(traveler_answers.get(column, "missing"))
        selections = re.findall(r"\d+", answer)

        if not selections:
            selections = ["missing"]

        encoded = loaded_encoders[column].transform([selections])
        feature_matrices.append(encoded)

    # Repeat the questionnaire features for every destination
    questionnaire_features = hstack(
        [
            csr_matrix(
                np.repeat(matrix.toarray(), len(destination_names), axis=0)
            )
            for matrix in feature_matrices
        ],
        format="csr",
        dtype=np.float32
    )

    # Encode all destination IDs in the original feature order
    destination_ids = np.arange(len(destination_names))

    destination_features = loaded_destination_encoder.transform(
        pd.DataFrame({"destination_id": destination_ids})
    )

    # Combine questionnaire and destination features
    X_candidates = hstack(
        [questionnaire_features, destination_features],
        format="csr",
        dtype=np.float32
    )

    # Predict calibrated acceptance probabilities
    probabilities = calibrated_model.predict_proba(X_candidates)[:, 1]

    results = pd.DataFrame({
        "destination_id": destination_ids,
        "destination": destination_names,
        "acceptance_probability": probabilities
    })

    results = results.sort_values(
        "acceptance_probability",
        ascending=False
    ).head(max(0, top_n))

    results["acceptance_probability"] = (
        results["acceptance_probability"] * 100
    ).round(2)

    return results.reset_index(drop=True)

In [12]:
import joblib
from pathlib import Path

calibrated_path = (
    Path("models") / "trip_recommendation_calibrated.joblib"
)

calibrated_artifacts = joblib.load(calibrated_path)

calibrated_model = calibrated_artifacts["model"]
loaded_encoders = calibrated_artifacts["encoders"]
loaded_destination_encoder = calibrated_artifacts["destination_encoder"]
loaded_feature_columns = calibrated_artifacts["feature_columns"]
loaded_threshold = calibrated_artifacts["threshold"]

print("Calibrated model loaded successfully!")
print(type(calibrated_model).__name__)

Calibrated model loaded successfully!
CalibratedClassifierCV


In [13]:
sample_traveler = df.iloc[0]

traveler_answers = {
    column: sample_traveler[column]
    for column in loaded_feature_columns
}

recommendations = recommend_destinations(
    traveler_answers,
    top_n=5
)

print(recommendations.to_string(index=False))

 destination_id             destination  acceptance_probability
            471            Kyoto, Japan                   73.66
            433           Amalfi, Italy                   71.69
            473            Nikko, Japan                   68.97
            459         Vernazza, Italy                   67.96
            432 Taormina, Sicily, Italy                   67.57


In [14]:
# Select one traveler's questionnaire responses
sample_traveler = df.iloc[0]

traveler_answers = {
    column: sample_traveler[column]
    for column in loaded_feature_columns
}

# Generate the top 5 recommendations
recommendations = recommend_destinations(
    traveler_answers,
    top_n=5
)

print("Top 5 Recommended Destinations:\n")
print(recommendations.to_string(index=False))

Top 5 Recommended Destinations:

 destination_id             destination  acceptance_probability
            471            Kyoto, Japan                   73.66
            433           Amalfi, Italy                   71.69
            473            Nikko, Japan                   68.97
            459         Vernazza, Italy                   67.96
            432 Taormina, Sicily, Italy                   67.57


In [15]:
# Select a different traveler
sample_traveler_2 = df.iloc[100]

traveler_answers_2 = {
    column: sample_traveler_2[column]
    for column in loaded_feature_columns
}

recommendations_2 = recommend_destinations(
    traveler_answers_2,
    top_n=5
)

print("Second traveler's recommendations:\n")
print(recommendations_2.to_string(index=False))

Second traveler's recommendations:

 destination_id             destination  acceptance_probability
            471            Kyoto, Japan                   74.96
            433           Amalfi, Italy                   73.05
            473            Nikko, Japan                   70.39
            459         Vernazza, Italy                   69.44
            432 Taormina, Sicily, Italy                   69.08


In [16]:
# Compare questionnaire responses for the two travelers
for column in loaded_feature_columns:
    print(
        f"{column}: "
        f"Traveler 1 = {sample_traveler[column]} | "
        f"Traveler 2 = {sample_traveler_2[column]}"
    )

form_a: Traveler 1 = ['0', '1'] | Traveler 2 = ['1']
form_b: Traveler 1 = ['1'] | Traveler 2 = ['2']
form_c: Traveler 1 = ['2'] | Traveler 2 = ['2']
form_f: Traveler 1 = ['0', '1', '2', '7'] | Traveler 2 = ['0', '1', '2']
form_g: Traveler 1 = ['0', '2', '4'] | Traveler 2 = ['1', '2', '4', '6', '7']
form_h: Traveler 1 = ['1'] | Traveler 2 = ['1']
form_i: Traveler 1 = ['1'] | Traveler 2 = ['1']
form_j: Traveler 1 = ['1'] | Traveler 2 = ['2']
form_k: Traveler 1 = ['2'] | Traveler 2 = ['0']


In [17]:
# Compare the same destinations for two travelers

destination_ids_to_test = [663, 471, 433, 473, 466]

def score_selected_destinations(traveler_answers, destination_ids):
    feature_matrices = []

    for column in loaded_feature_columns:
        answer = str(traveler_answers.get(column, "missing"))
        selections = re.findall(r"\d+", answer)

        if not selections:
            selections = ["missing"]

        encoded = loaded_encoders[column].transform([selections])
        feature_matrices.append(encoded)

    questionnaire_features = hstack(
        feature_matrices, format="csr", dtype=np.float32
    )

    destination_features = loaded_destination_encoder.transform(
        pd.DataFrame({"destination_id": destination_ids})
    )

    repeated_questionnaire = hstack(
        [
            csr_matrix(
                np.repeat(matrix.toarray(), len(destination_ids), axis=0)
            )
            for matrix in feature_matrices
        ],
        format="csr",
        dtype=np.float32
    )

    X_candidates = hstack(
        [repeated_questionnaire, destination_features],
        format="csr",
        dtype=np.float32
    )

    return loaded_model.predict_proba(X_candidates)[:, 1] * 100


scores_1 = score_selected_destinations(
    traveler_answers, destination_ids_to_test
)

scores_2 = score_selected_destinations(
    traveler_answers_2, destination_ids_to_test
)

comparison = pd.DataFrame({
    "destination": [destination_names[i] for i in destination_ids_to_test],
    "traveler_1_score": scores_1.round(2),
    "traveler_2_score": scores_2.round(2),
})

comparison["score_difference"] = (
    comparison["traveler_2_score"] - comparison["traveler_1_score"]
).round(2)

print(comparison.to_string(index=False))

       destination  traveler_1_score  traveler_2_score  score_difference
  Lisbon, Portugal         85.120003         86.040001              0.92
      Kyoto, Japan         83.139999         84.160004              1.02
     Amalfi, Italy         82.500000         83.550003              1.05
      Nikko, Japan         80.510002         81.650002              1.14
Ocho Rios, Jamaica         79.739998         80.919998              1.18


In [18]:
from pathlib import Path

for path in Path("data").iterdir():
    print(path.name)

clean_training_data.csv
destination_features.csv
destination_ids.txt
external
matched_attractions.csv
Stravl_Travel_Preference_Data.csv
training_data.csv


In [19]:
import numpy as np

feature_names = []

for column in loaded_feature_columns:
    encoder = loaded_encoders[column]
    feature_names.extend(
        [f"{column}={value}" for value in encoder.classes_]
    )

destination_feature_names = [
    f"destination_id={destination_id}"
    for destination_id in loaded_destination_encoder.categories_[0]
]

all_feature_names = feature_names + destination_feature_names
coefficients = loaded_model.coef_[0]

feature_importance = pd.DataFrame({
    "feature": all_feature_names,
    "coefficient": coefficients
})

feature_importance["absolute_coefficient"] = (
    feature_importance["coefficient"].abs()
)

print("Top 15 features by absolute coefficient:\n")
print(
    feature_importance
    .sort_values("absolute_coefficient", ascending=False)
    .head(15)[["feature", "coefficient"]]
    .to_string(index=False)
)

Top 15 features by absolute coefficient:

            feature  coefficient
destination_id=1069    -2.275741
destination_id=1074    -2.252238
destination_id=1110    -2.220201
destination_id=1090    -2.176944
destination_id=1196    -2.139539
destination_id=1101    -2.097357
destination_id=1092    -2.008994
destination_id=1086    -1.971355
destination_id=1091    -1.971255
 destination_id=866    -1.957741
destination_id=1384    -1.919778
destination_id=1082    -1.908670
 destination_id=663     1.902133
 destination_id=869    -1.897641
destination_id=1068    -1.885688


In [20]:
# Separate questionnaire features from destination features

questionnaire_importance = feature_importance[
    ~feature_importance["feature"].str.startswith("destination_id=")
].copy()

print("Top 15 questionnaire features:\n")

print(
    questionnaire_importance
    .sort_values("absolute_coefficient", ascending=False)
    .head(15)[["feature", "coefficient"]]
    .to_string(index=False)
)

print("\nAverage absolute coefficient:")

print(
    "Questionnaire features:",
    round(questionnaire_importance["absolute_coefficient"].mean(), 4)
)

destination_importance = feature_importance[
    feature_importance["feature"].str.startswith("destination_id=")
]

print(
    "Destination features:",
    round(destination_importance["absolute_coefficient"].mean(), 4)
)


Top 15 questionnaire features:

       feature  coefficient
form_c=missing     1.849743
form_a=missing     1.199218
form_h=missing     1.199218
form_f=missing     0.807281
      form_c=1    -0.592018
form_k=missing    -0.568639
form_i=missing    -0.568639
form_b=missing    -0.568639
form_j=missing    -0.568639
      form_h=0    -0.565177
      form_c=0    -0.543223
      form_c=2    -0.518754
      form_c=3    -0.484161
      form_h=2    -0.476281
      form_h=1    -0.463855

Average absolute coefficient:
Questionnaire features: 0.3227
Destination features: 0.7194


In [21]:
# Check missing-value representation in questionnaire fields

for column in loaded_feature_columns:
    missing_count = (
        df[column].astype(str).eq("missing").sum()
    )

    print(
        f"{column}: {missing_count:,} missing "
        f"({missing_count / len(df) * 100:.2f}%)"
    )

form_a: 10 missing (0.00%)
form_b: 20 missing (0.00%)
form_c: 30 missing (0.00%)
form_f: 3,722 missing (0.47%)
form_g: 10,509 missing (1.34%)
form_h: 10 missing (0.00%)
form_i: 20 missing (0.00%)
form_j: 20 missing (0.00%)
form_k: 20 missing (0.00%)


In [22]:
# Calculate historical acceptance rate for each destination

destination_stats = (
    df.groupby(["destination_id", "destination"])["accepted"]
      .agg(["count", "mean"])
      .reset_index()
)

destination_stats = destination_stats.rename(
    columns={
        "count": "total_responses",
        "mean": "acceptance_rate"
    }
)

destination_stats["acceptance_rate"] *= 100

print("Destinations in cleaned dataset:", len(destination_stats))

print("\nHighest historical acceptance rates (minimum 100 responses):")
print(
    destination_stats[destination_stats["total_responses"] >= 100]
    .sort_values("acceptance_rate", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\nLowest historical acceptance rates (minimum 100 responses):")
print(
    destination_stats[destination_stats["total_responses"] >= 100]
    .sort_values("acceptance_rate", ascending=True)
    .head(10)
    .to_string(index=False)
)

Destinations in cleaned dataset: 1452

Highest historical acceptance rates (minimum 100 responses):
 destination_id             destination  total_responses  acceptance_rate
            471            Kyoto, Japan              937        73.532551
            459         Vernazza, Italy              419        72.553699
            433           Amalfi, Italy              438        72.146119
            473            Nikko, Japan              414        71.014493
            363        Skiathos, Greece              347        69.740634
            658       Cascais, Portugal              389        68.894602
            662         Lagos, Portugal              372        68.279570
            367       Zakynthos, Greece              486        67.489712
            430  Palermo, Sicily, Italy              375        67.466667
            781 Interlaken, Switzerland              311        67.202572

Lowest historical acceptance rates (minimum 100 responses):
 destination_id          

In [23]:
# Check the number of responses per traveler

traveler_stats = (
    df.groupby("traveler_id")["destination_id"]
      .agg(["count", "nunique"])
      .rename(columns={
          "count": "response_count",
          "nunique": "unique_destinations"
      })
)

print("Total travelers:", len(traveler_stats))
print("\nResponses per traveler:")
print(traveler_stats["response_count"].describe())

print("\nTravelers with at least 5 destination responses:",
      (traveler_stats["response_count"] >= 5).sum())

print("Travelers with at least 10 destination responses:",
      (traveler_stats["response_count"] >= 10).sum())

Total travelers: 60886

Responses per traveler:
count    60886.000000
mean        12.928062
std         16.848952
min          1.000000
25%         10.000000
50%         10.000000
75%         10.000000
max        630.000000
Name: response_count, dtype: float64

Travelers with at least 5 destination responses: 60379
Travelers with at least 10 destination responses: 53971


In [24]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Training examples:", len(train_idx))
print("Testing examples:", len(test_idx))
print(
    "Overlapping travelers:",
    len(set(groups[train_idx]) & set(groups[test_idx]))
)

Training examples: 630586
Testing examples: 156552
Overlapping travelers: 0


In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.sparse import issparse
import time

# Use a manageable training sample for the first experiment
rng = np.random.default_rng(42)
sample_size = min(100_000, len(train_idx))
sample_positions = rng.choice(
    len(train_idx),
    size=sample_size,
    replace=False
)

X_rf_train = X_train[sample_positions]
y_rf_train = y_train[sample_positions]

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

start = time.time()
rf_model.fit(X_rf_train, y_rf_train)

print("Random Forest training completed!")
print("Training rows:", len(y_rf_train))
print(f"Training time: {time.time() - start:.2f} seconds")

Random Forest training completed!
Training rows: 100000
Training time: 3.77 seconds


In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
import time

start = time.time()

rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest evaluation completed!")
print(f"Evaluation time: {time.time() - start:.2f} seconds")

print("\nAccuracy:", round(accuracy_score(y_test, rf_predictions), 4))
print("Precision:", round(precision_score(y_test, rf_predictions), 4))
print("Recall:", round(recall_score(y_test, rf_predictions), 4))
print("F1-score:", round(f1_score(y_test, rf_predictions), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_probabilities), 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_predictions,
        target_names=["Rejected", "Accepted"]
    )
)

Random Forest evaluation completed!
Evaluation time: 0.71 seconds

Accuracy: 0.5849
Precision: 0.4465
Recall: 0.5487
F1-score: 0.4923
ROC-AUC: 0.6069

Classification Report:
              precision    recall  f1-score   support

    Rejected       0.70      0.61      0.65     99120
    Accepted       0.45      0.55      0.49     57432

    accuracy                           0.58    156552
   macro avg       0.57      0.58      0.57    156552
weighted avg       0.61      0.58      0.59    156552



In [27]:
# Confirm the interaction data is available for collaborative filtering

print("Training interaction rows:", len(train_idx))
print("Testing interaction rows:", len(test_idx))

print(
    "Unique training travelers:",
    len(set(groups[train_idx]))
)

print(
    "Unique training destinations:",
    len(set(df.iloc[train_idx]["destination_id"]))
)

print(
    "Unique test destinations:",
    len(set(df.iloc[test_idx]["destination_id"]))
)

Training interaction rows: 630586
Testing interaction rows: 156552
Unique training travelers: 48708
Unique training destinations: 1448
Unique test destinations: 1442


In [28]:
# Build destination acceptance rates using training data only

train_interactions = df.iloc[train_idx][
    ["destination_id", "accepted"]
].copy()

destination_popularity = (
    train_interactions
    .groupby("destination_id")["accepted"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

destination_popularity = destination_popularity.rename(
    columns={
        "sum": "accepted_count",
        "count": "response_count",
        "mean": "acceptance_rate"
    }
)

destination_popularity = destination_popularity.sort_values(
    "acceptance_rate",
    ascending=False
)

print("Popularity baseline created!")
print("\nTop 10 destinations from training data:")

print(
    destination_popularity
    .head(10)
    .assign(
        acceptance_rate=lambda x: (
            x["acceptance_rate"] * 100
        ).round(2)
    )
    .to_string(index=False)
)

Popularity baseline created!

Top 10 destinations from training data:
 destination_id  accepted_count  response_count  acceptance_rate
           1329               1               1           100.00
            772               1               1           100.00
            670               1               1           100.00
            495               1               1           100.00
            485               1               1           100.00
            663               8               9            88.89
            789               6               8            75.00
            471             557             748            74.47
            595              26              35            74.29
            433             256             352            72.73


In [29]:
# Calculate the overall training acceptance rate
overall_rate = train_interactions["accepted"].mean()

# Smooth destination acceptance rates
smoothing_strength = 20

destination_popularity["smoothed_rate"] = (
    destination_popularity["accepted_count"]
    + smoothing_strength * overall_rate
) / (
    destination_popularity["response_count"]
    + smoothing_strength
)

destination_popularity = destination_popularity.sort_values(
    "smoothed_rate",
    ascending=False
)

print(f"Overall training acceptance rate: {overall_rate:.2%}")
print("\nTop 10 destinations by smoothed acceptance rate:")

print(
    destination_popularity[
        [
            "destination_id",
            "accepted_count",
            "response_count",
            "smoothed_rate"
        ]
    ]
    .head(10)
    .assign(
        smoothed_rate=lambda x: (
            x["smoothed_rate"] * 100
        ).round(2)
    )
    .to_string(index=False)
)

Overall training acceptance rate: 36.70%

Top 10 destinations by smoothed acceptance rate:
 destination_id  accepted_count  response_count  smoothed_rate
            471             557             748          73.48
            433             256             352          70.79
            459             241             340          68.98
            473             227             321          68.72
            354             305             446          67.03
            474             936            1389          66.95
            478             531             788          66.63
            367             266             393          66.18
            363             183             268          66.09
            387             663             995          66.04


In [30]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Actual labels from the test set
y_true = y_test

# Logistic Regression predictions already calculated earlier
logistic_probabilities = model.predict_proba(X_test)[:, 1]

# Popularity scores for test destinations
popularity_scores = (
    df.iloc[test_idx]["destination_id"]
    .map(
        destination_popularity.set_index("destination_id")[
            "smoothed_rate"
        ]
    )
    .fillna(overall_rate)
    .to_numpy()
)

# Evaluate both approaches
print("LOGISTIC REGRESSION")
print("ROC-AUC:", round(roc_auc_score(y_true, logistic_probabilities), 4))
print(
    "Average Precision:",
    round(average_precision_score(y_true, logistic_probabilities), 4)
)

print("\nPOPULARITY BASELINE")
print("ROC-AUC:", round(roc_auc_score(y_true, popularity_scores), 4))
print(
    "Average Precision:",
    round(average_precision_score(y_true, popularity_scores), 4)
)

LOGISTIC REGRESSION
ROC-AUC: 0.7216
Average Precision: 0.581

POPULARITY BASELINE
ROC-AUC: 0.7029
Average Precision: 0.5432


In [31]:
# Compare recommendations for two different travelers

answers_1 = {
    column: df.iloc[0][column]
    for column in loaded_feature_columns
}

answers_2 = {
    column: df.iloc[100][column]
    for column in loaded_feature_columns
}

recommendations_1 = recommend_destinations(answers_1, top_n=10)
recommendations_2 = recommend_destinations(answers_2, top_n=10)

print("TRAVELER 1 RECOMMENDATIONS")
print(recommendations_1.to_string(index=False))

print("\nTRAVELER 2 RECOMMENDATIONS")
print(recommendations_2.to_string(index=False))

TRAVELER 1 RECOMMENDATIONS
 destination_id             destination  acceptance_probability
            471            Kyoto, Japan                   73.66
            433           Amalfi, Italy                   71.69
            473            Nikko, Japan                   68.97
            459         Vernazza, Italy                   67.96
            432 Taormina, Sicily, Italy                   67.57
            354             Kos, Greece                   66.92
            367       Zakynthos, Greece                   66.59
            658       Cascais, Portugal                   66.08
            662         Lagos, Portugal                   66.03
            474            Osaka, Japan                   66.03

TRAVELER 2 RECOMMENDATIONS
 destination_id             destination  acceptance_probability
            471            Kyoto, Japan                   74.96
            433           Amalfi, Italy                   73.05
            473            Nikko, Japan          

In [32]:
# Compare questionnaire answers for two travelers

traveler_1 = df.iloc[0]
traveler_2 = df.iloc[100]

print("QUESTIONNAIRE COMPARISON\n")

for column in loaded_feature_columns:
    answer_1 = traveler_1[column]
    answer_2 = traveler_2[column]

    print(f"{column}:")
    print(f"  Traveler 1: {answer_1}")
    print(f"  Traveler 2: {answer_2}")
    print(f"  Same answers: {answer_1 == answer_2}")
    print()

QUESTIONNAIRE COMPARISON

form_a:
  Traveler 1: ['0', '1']
  Traveler 2: ['1']
  Same answers: False

form_b:
  Traveler 1: ['1']
  Traveler 2: ['2']
  Same answers: False

form_c:
  Traveler 1: ['2']
  Traveler 2: ['2']
  Same answers: True

form_f:
  Traveler 1: ['0', '1', '2', '7']
  Traveler 2: ['0', '1', '2']
  Same answers: False

form_g:
  Traveler 1: ['0', '2', '4']
  Traveler 2: ['1', '2', '4', '6', '7']
  Same answers: False

form_h:
  Traveler 1: ['1']
  Traveler 2: ['1']
  Same answers: True

form_i:
  Traveler 1: ['1']
  Traveler 2: ['1']
  Same answers: True

form_j:
  Traveler 1: ['1']
  Traveler 2: ['2']
  Same answers: False

form_k:
  Traveler 1: ['2']
  Traveler 2: ['0']
  Same answers: False



In [33]:
import pandas as pd
import numpy as np

all_destination_ids = np.arange(len(destination_names))

scores_1 = score_selected_destinations(
    answers_1, all_destination_ids
)

scores_2 = score_selected_destinations(
    answers_2, all_destination_ids
)

# Convert arrays into DataFrames
comparison = pd.DataFrame({
    "destination_id": all_destination_ids,
    "score_traveler1": np.asarray(scores_1).flatten(),
    "score_traveler2": np.asarray(scores_2).flatten()
})

# Calculate absolute score differences
comparison["score_difference"] = (
    comparison["score_traveler2"]
    - comparison["score_traveler1"]
).abs()

print(
    "Average absolute score difference:",
    round(comparison["score_difference"].mean(), 4)
)

print(
    "Maximum absolute score difference:",
    round(comparison["score_difference"].max(), 4)
)

print("\nDestinations with the largest score differences:")

print(
    comparison.nlargest(10, "score_difference").to_string(index=False)
)

Average absolute score difference: 1.6043
Maximum absolute score difference: 1.8684

Destinations with the largest score differences:
 destination_id  score_traveler1  score_traveler2  score_difference
            593        49.109501        50.977940          1.868439
            812        49.074017        50.942455          1.868439
            113        48.937271        50.805698          1.868427
            157        48.918770        50.787197          1.868427
            636        49.252316        51.120728          1.868412
           1333        48.836082        50.704487          1.868404
           1340        49.296581        51.164978          1.868397
            511        48.803776        50.672169          1.868393
            592        49.365288        51.233662          1.868374
            250        49.373558        51.241928          1.868370


In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Use destination ID features only
X_destination = destination_features

# Use the same train/test split
X_dest_train = X_destination[train_idx]
X_dest_test = X_destination[test_idx]

# Train the destination-only model
destination_only_model = LogisticRegression(
    max_iter=200,
    solver="saga",
    class_weight="balanced",
    random_state=42
)

destination_only_model.fit(X_dest_train, y_train)

# Evaluate both models on the same test rows
destination_only_probs = destination_only_model.predict_proba(
    X_dest_test
)[:, 1]

full_model_probs = model.predict_proba(X_test)[:, 1]

print("FULL MODEL (questionnaire + destination)")
print("ROC-AUC:", round(roc_auc_score(y_test, full_model_probs), 4))
print("Average Precision:", round(
    average_precision_score(y_test, full_model_probs), 4
))

print("\nDESTINATION-ONLY MODEL")
print("ROC-AUC:", round(roc_auc_score(y_test, destination_only_probs), 4))
print("Average Precision:", round(
    average_precision_score(y_test, destination_only_probs), 4
))

FULL MODEL (questionnaire + destination)
ROC-AUC: 0.7216
Average Precision: 0.581

DESTINATION-ONLY MODEL
ROC-AUC: 0.7029
Average Precision: 0.5433


In [35]:
# Inspect accepted destinations for the two sample travelers

for traveler_index in [0, 100]:
    traveler_id = df.iloc[traveler_index]["traveler_id"]

    traveler_rows = df[df["traveler_id"] == traveler_id]

    accepted_rows = traveler_rows[traveler_rows["accepted"] == 1]
    rejected_rows = traveler_rows[traveler_rows["accepted"] == 0]

    print(f"\nTraveler ID: {traveler_id}")
    print("Total responses:", len(traveler_rows))
    print("Accepted destinations:", len(accepted_rows))
    print("Rejected destinations:", len(rejected_rows))

    print("\nExample accepted destinations:")
    print(
        accepted_rows[["destination_id", "accepted"]]
        .head(10)
        .to_string(index=False)
    )


Traveler ID: 22871c2cb8ee11ec9a19fefb5d5b7ead
Total responses: 20
Accepted destinations: 9
Rejected destinations: 11

Example accepted destinations:
 destination_id  accepted
            837         1
            779         1
           1315         1
            773         1
            621         1
            799         1
            794         1
            172         1
            190         1

Traveler ID: 72cb8a02b91011ecb3ce0274c9fee463
Total responses: 50
Accepted destinations: 4
Rejected destinations: 46

Example accepted destinations:
 destination_id  accepted
           1181         1
           1316         1
           1366         1
            581         1


In [36]:
import numpy as np
import pandas as pd

# Use the held-out test data
test_data = df.iloc[test_idx].copy()

# Get model scores for every test interaction
test_data["score"] = model.predict_proba(X_test)[:, 1]

# Rank destinations separately for each traveler
test_data = test_data.sort_values(
    ["traveler_id", "score"],
    ascending=[True, False]
)

# Evaluate only travelers with at least one accepted destination
recall_at_5 = []
recall_at_10 = []

for traveler_id, group in test_data.groupby("traveler_id"):
    relevant = set(
        group.loc[group["accepted"] == 1, "destination_id"]
    )

    if not relevant:
        continue

    ranked_destinations = group["destination_id"].tolist()

    top_5 = set(ranked_destinations[:5])
    top_10 = set(ranked_destinations[:10])

    recall_at_5.append(len(relevant & top_5) / len(relevant))
    recall_at_10.append(len(relevant & top_10) / len(relevant))

print("Test travelers evaluated:", len(recall_at_5))
print("Mean Recall@5:", round(np.mean(recall_at_5), 4))
print("Mean Recall@10:", round(np.mean(recall_at_10), 4))

Test travelers evaluated: 9438
Mean Recall@5: 0.649
Mean Recall@10: 0.9365


In [37]:
# Select 100 test travelers
sample_test_travelers = (
    test_data["traveler_id"].drop_duplicates().head(100).tolist()
)

print("Travelers selected:", len(sample_test_travelers))
print("Total destinations available:", len(destination_names))

Travelers selected: 100
Total destinations available: 1486


In [38]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

def recommend_unseen_destinations(traveler_answers, seen_ids, top_n=10):
    destination_ids = np.arange(len(destination_names))

    # Encode the traveler's questionnaire answers
    feature_matrices = []

    for column in loaded_feature_columns:
        answer = str(traveler_answers.get(column, "missing"))
        selections = re.findall(r"\d+", answer)

        if not selections:
            selections = ["missing"]

        feature_matrices.append(
            loaded_encoders[column].transform([selections])
        )

    # Repeat questionnaire features for every destination
    repeated_questionnaire = hstack(
        [
            csr_matrix(
                np.repeat(matrix.toarray(), len(destination_ids), axis=0)
            )
            for matrix in feature_matrices
        ],
        format="csr",
        dtype=np.float32
    )

    # Encode destinations and combine features
    destination_features_all = loaded_destination_encoder.transform(
        pd.DataFrame({"destination_id": destination_ids})
    )

    X_candidates = hstack(
        [repeated_questionnaire, destination_features_all],
        format="csr",
        dtype=np.float32
    )

    # Predict acceptance probabilities
    scores = loaded_model.predict_proba(X_candidates)[:, 1]

    # Exclude destinations the traveler has already responded to
    unseen_mask = ~np.isin(destination_ids, list(seen_ids))

    results = pd.DataFrame({
        "destination_id": destination_ids[unseen_mask],
        "destination": np.array(destination_names)[unseen_mask],
        "score": scores[unseen_mask]
    })

    return results.nlargest(top_n, "score").reset_index(drop=True)


print("Recommendation function created successfully.")

Recommendation function created successfully.


In [39]:
recall_5 = []
recall_10 = []
train_df = df.iloc[train_idx]

for traveler_id in sample_test_travelers:

    # This traveler's held-out test interactions
    traveler_test = test_data[
        test_data["traveler_id"] == traveler_id
    ]

    # Destinations accepted in the test data
    relevant_destinations = set(
        traveler_test.loc[
            traveler_test["accepted"] == 1,
            "destination_id"
        ]
    )

    if not relevant_destinations:
        continue

   # Exclude only destinations in training data
    seen_destinations = set(
    train_df.loc[
        train_df["traveler_id"] == traveler_id,
        "destination_id"
        ]
    )

    # Use questionnaire answers from this traveler
    traveler_row = df.loc[
        df["traveler_id"] == traveler_id
    ].iloc[0]

    traveler_answers = {
        column: traveler_row[column]
        for column in loaded_feature_columns
    }

    # Recommend unseen destinations
    recommendations = recommend_unseen_destinations(
        traveler_answers,
        seen_destinations,
        top_n=10
    )

    top_5 = set(recommendations.head(5)["destination_id"])
    top_10 = set(recommendations["destination_id"])

    recall_5.append(
        len(relevant_destinations & top_5)
        / len(relevant_destinations)
    )

    recall_10.append(
        len(relevant_destinations & top_10)
        / len(relevant_destinations)
    )

print("Travelers evaluated:", len(recall_5))
print("Full-catalog Recall@5:", round(np.mean(recall_5), 4))
print("Full-catalog Recall@10:", round(np.mean(recall_10), 4))

Travelers evaluated: 82
Full-catalog Recall@5: 0.0
Full-catalog Recall@10: 0.0041


In [40]:
# Inspect the first evaluated traveler
traveler_id = sample_test_travelers[0]

traveler_test = test_data[
    test_data["traveler_id"] == traveler_id
]

relevant_destinations = set(
    traveler_test.loc[
        traveler_test["accepted"] == 1,
        "destination_id"
    ]
)

seen_destinations = set(
    df.loc[
        df["traveler_id"] == traveler_id,
        "destination_id"
    ]
)

traveler_row = df.loc[
    df["traveler_id"] == traveler_id
].iloc[0]

traveler_answers = {
    column: traveler_row[column]
    for column in loaded_feature_columns
}

recommendations = recommend_unseen_destinations(
    traveler_answers,
    seen_destinations,
    top_n=10
)

print("Traveler ID:", traveler_id)
print("Accepted test destinations:", relevant_destinations)
print("Previously seen destinations:", len(seen_destinations))

print("\nTop 10 unseen recommendations:")
print(recommendations.to_string(index=False))

print("\nAccepted test destinations in top 10:")
print(
    recommendations[
        recommendations["destination_id"].isin(relevant_destinations)
    ].to_string(index=False)
)

Traveler ID: 0003d1f6b95f11ecafb25ada508e54ec
Accepted test destinations: {819}
Previously seen destinations: 10

Top 10 unseen recommendations:
 destination_id        destination    score
            663   Lisbon, Portugal 0.822747
            471       Kyoto, Japan 0.800048
            433      Amalfi, Italy 0.792758
            473       Nikko, Japan 0.770177
            466 Ocho Rios, Jamaica 0.761505
            459    Vernazza, Italy 0.760282
            595    Ålesund, Norway 0.755911
            354        Kos, Greece 0.748312
            367  Zakynthos, Greece 0.745128
            363   Skiathos, Greece 0.744887

Accepted test destinations in top 10:
Empty DataFrame
Columns: [destination_id, destination, score]
Index: []


In [41]:
# Score all unseen destinations, not just the top 10
all_recommendations = recommend_unseen_destinations(
    traveler_answers,
    seen_destinations,
    top_n=len(destination_names)
)

target_id = 819

target_result = all_recommendations[
    all_recommendations["destination_id"] == target_id
]

if target_result.empty:
    print("Destination 819 is not available in the unseen candidates.")
else:
    target_rank = (
        all_recommendations["score"].rank(
            ascending=False,
            method="min"
        )[target_result.index[0]]
    )

    print("Accepted destination ID:", target_id)
    print(
        "Destination name:",
        all_recommendations.loc[
            all_recommendations["destination_id"] == target_id,
            "destination"
        ].iloc[0]
    )
    print(
        "Predicted acceptance score:",
        round(target_result["score"].iloc[0], 4)
    )
    print("Rank among unseen destinations:", int(target_rank))

Destination 819 is not available in the unseen candidates.


In [42]:
# Check whether test destinations are represented in the model

known_destination_ids = set(
    loaded_destination_encoder.categories_[0]
)

test_destination_ids = set(
    test_data["destination_id"].unique()
)

accepted_test_ids = set(
    test_data.loc[
        test_data["accepted"] == 1,
        "destination_id"
    ].unique()
)

print("Destinations known to the model:", len(known_destination_ids))
print("Destinations in test data:", len(test_destination_ids))
print("Accepted test destinations:", len(accepted_test_ids))

print(
    "Accepted destinations known to the model:",
    len(accepted_test_ids & known_destination_ids)
)

print(
    "Accepted destinations missing from the model:",
    len(accepted_test_ids - known_destination_ids)
)

Destinations known to the model: 1452
Destinations in test data: 1442
Accepted test destinations: 1439
Accepted destinations known to the model: 1439
Accepted destinations missing from the model: 0


In [43]:
# Inspect the earlier example traveler
traveler_id = "0003d1f6b95f11ecafb25ada508e54ec"

traveler_row = df.loc[
    df["traveler_id"] == traveler_id
].iloc[0]

traveler_answers = {
    column: traveler_row[column]
    for column in loaded_feature_columns
}

# Score destination 819 directly
target_scores = score_selected_destinations(
    traveler_answers,
    [819]
)

print("Traveler ID:", traveler_id)
print("Accepted destination ID: 819")
print("Predicted acceptance score:", target_scores)

Traveler ID: 0003d1f6b95f11ecafb25ada508e54ec
Accepted destination ID: 819
Predicted acceptance score: [54.53218]


In [44]:
print("Destination 819:", destination_names[819])

Destination 819: Ephesus, Turkey


In [45]:
# Inspect training interactions for Ephesus, Turkey (destination 819)

ephesus_data = train_df[
    train_df["destination_id"] == 819
]

print("Destination: Ephesus, Turkey")
print("Training responses:", len(ephesus_data))

print("\nAcceptance counts:")
print(ephesus_data["accepted"].value_counts().sort_index())

print("\nAcceptance rate:")
if len(ephesus_data) > 0:
    print(f"{ephesus_data['accepted'].mean():.2%}")
else:
    print("No training interactions for this destination.")

Destination: Ephesus, Turkey
Training responses: 295

Acceptance counts:
accepted
0    156
1    139
Name: count, dtype: int64

Acceptance rate:
47.12%


In [46]:
# Compare Ephesus predictions for different travelers

sample_travelers = df.drop_duplicates("traveler_id").head(10)

comparison = []

for _, row in sample_travelers.iterrows():
    answers = {
        column: row[column]
        for column in loaded_feature_columns
    }

    score = score_selected_destinations(answers, [819])[0]

    comparison.append({
        "traveler_id": row["traveler_id"],
        "ephesus_score": round(float(score), 2)
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df.to_string(index=False))
print("\nScore range:",
      round(comparison_df["ephesus_score"].max()
            - comparison_df["ephesus_score"].min(), 2),
      "percentage points")

                     traveler_id  ephesus_score
22871c2cb8ee11ec9a19fefb5d5b7ead          59.65
89b7022ab8f011ec9a19fefb5d5b7ead          66.04
c64f6c36b8f011ec9a19fefb5d5b7ead          71.81
1eb6079ab8f111ec9a19fefb5d5b7ead          52.51
285e9852b8f111ec9a19fefb5d5b7ead          45.87
900566fcb8f111ec9a19fefb5d5b7ead          68.17
72cb8a02b91011ecb3ce0274c9fee463          61.43
6757e288b91011ecb3ce0274c9fee463          72.25
46c9cbc2b8f111ec9a19fefb5d5b7ead          48.46
9cbff330b8f111ec9a19fefb5d5b7ead          65.02

Score range: 26.38 percentage points


In [47]:
# Evaluate Ephesus predictions on held-out test travelers

ephesus_test = df.iloc[test_idx]
ephesus_test = ephesus_test[
    ephesus_test["destination_id"] == 819
].copy()

print("Ephesus test responses:", len(ephesus_test))

if len(ephesus_test) > 0:
    actual_acceptance_rate = ephesus_test["accepted"].mean()
    print(f"Actual acceptance rate: {actual_acceptance_rate:.2%}")

    predictions = []

    for _, row in ephesus_test.iterrows():
        answers = {
            column: row[column]
            for column in loaded_feature_columns
        }

        score = score_selected_destinations(answers, [819])[0]
        predictions.append(float(score) / 100)

    ephesus_test["predicted_score"] = predictions

    print(
        f"Average predicted score: "
        f"{ephesus_test['predicted_score'].mean():.2%}"
    )

    print("\nActual outcomes:")
    print(ephesus_test["accepted"].value_counts().sort_index())
else:
    print("No Ephesus responses found in the test set.")

Ephesus test responses: 60
Actual acceptance rate: 40.00%
Average predicted score: 58.33%

Actual outcomes:
accepted
0    36
1    24
Name: count, dtype: int64


In [48]:
from sklearn.calibration import calibration_curve
import pandas as pd
import numpy as np

y_true = ephesus_test["accepted"].to_numpy()
y_prob = ephesus_test["predicted_score"].to_numpy()

calibration_table = pd.DataFrame({
    "predicted_score": y_prob,
    "actual_acceptance": y_true
})

calibration_table["score_group"] = pd.cut(
    calibration_table["predicted_score"],
    bins=[0, 0.4, 0.5, 0.6, 0.7, 1.0],
    include_lowest=True
)

summary = calibration_table.groupby(
    "score_group", observed=False
).agg(
    responses=("actual_acceptance", "size"),
    average_predicted=("predicted_score", "mean"),
    actual_acceptance_rate=("actual_acceptance", "mean")
).round(3)

print(summary)

               responses  average_predicted  actual_acceptance_rate
score_group                                                        
(-0.001, 0.4]          0                NaN                     NaN
(0.4, 0.5]             9              0.471                   0.111
(0.5, 0.6]            29              0.557                   0.379
(0.6, 0.7]            16              0.644                   0.562
(0.7, 1.0]             6              0.714                   0.500


In [49]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import numpy as np
import pandas as pd

# Predict acceptance probabilities for all held-out test rows
test_probabilities = model.predict_proba(X_test)[:, 1]

# Compare predictions against actual outcomes
print("Overall Brier score:",
      round(brier_score_loss(y_test, test_probabilities), 4))

actual_rates, predicted_rates = calibration_curve(
    y_test,
    test_probabilities,
    n_bins=10,
    strategy="quantile"
)

calibration_summary = pd.DataFrame({
    "average_predicted": predicted_rates,
    "actual_acceptance_rate": actual_rates
}).round(3)

print("\nCalibration across all test destinations:")
print(calibration_summary.to_string(index=False))

Overall Brier score: 0.2142

Calibration across all test destinations:
 average_predicted  actual_acceptance_rate
             0.146                   0.116
             0.234                   0.155
             0.316                   0.204
             0.395                   0.268
             0.468                   0.325
             0.533                   0.386
             0.590                   0.443
             0.644                   0.515
             0.699                   0.580
             0.776                   0.678


In [50]:
from sklearn.metrics import brier_score_loss

# Use training data only to estimate the baseline acceptance rate
baseline_probability = y_train.mean()

baseline_predictions = np.full(
    len(y_test),
    baseline_probability
)

baseline_brier = brier_score_loss(
    y_test,
    baseline_predictions
)

model_brier = brier_score_loss(
    y_test,
    test_probabilities
)

print(f"Training acceptance rate: {baseline_probability:.4f}")
print(f"Baseline Brier score:     {baseline_brier:.4f}")
print(f"Model Brier score:        {model_brier:.4f}")
print(f"Improvement:              {(baseline_brier - model_brier):.4f}")

Training acceptance rate: 0.3670
Baseline Brier score:     0.2323
Model Brier score:        0.2142
Improvement:              0.0180


In [51]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

# Fit a fresh model on training data only
calibration_base_model = LogisticRegression(
    max_iter=200,
    solver="saga",
    class_weight="balanced",
    random_state=42
)

calibrated_model = CalibratedClassifierCV(
    estimator=calibration_base_model,
    method="sigmoid",
    cv=3,
    n_jobs=-1
)

calibrated_model.fit(X_train, y_train)

# Evaluate on the untouched test set
calibrated_probabilities = calibrated_model.predict_proba(X_test)[:, 1]

calibrated_brier = brier_score_loss(
    y_test,
    calibrated_probabilities
)

print(f"Original Brier score:   {model_brier:.4f}")
print(f"Calibrated Brier score: {calibrated_brier:.4f}")

Original Brier score:   0.2142
Calibrated Brier score: 0.2004


In [52]:
from sklearn.metrics import roc_auc_score, average_precision_score

print("Original model")
print(f"ROC-AUC:          {roc_auc_score(y_test, test_probabilities):.4f}")
print(f"Average Precision: {average_precision_score(y_test, test_probabilities):.4f}")

print("\nCalibrated model")
print(f"ROC-AUC:          {roc_auc_score(y_test, calibrated_probabilities):.4f}")
print(f"Average Precision: {average_precision_score(y_test, calibrated_probabilities):.4f}")

Original model
ROC-AUC:          0.7216
Average Precision: 0.5810

Calibrated model
ROC-AUC:          0.7212
Average Precision: 0.5800


In [53]:
import joblib
from pathlib import Path

model_path = Path("models") / "trip_recommendation_calibrated.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)

calibrated_artifacts = {
    "model": calibrated_model,
    "encoders": loaded_encoders,
    "destination_encoder": loaded_destination_encoder,
    "feature_columns": loaded_feature_columns,
    "threshold": loaded_threshold,
}

joblib.dump(calibrated_artifacts, model_path)

print("Calibrated model saved successfully!")
print("Location:", model_path.resolve())
print(f"File size: {model_path.stat().st_size / (1024 * 1024):.2f} MB")

Calibrated model saved successfully!
Location: C:\Users\Chandrashekar\OneDrive\Desktop\Place Recommandation\models\trip_recommendation_calibrated.joblib
File size: 0.03 MB


In [54]:
import joblib
from pathlib import Path

calibrated_path = (
    Path("models") / "trip_recommendation_calibrated.joblib"
)

calibrated_artifacts = joblib.load(calibrated_path)

calibrated_model = calibrated_artifacts["model"]
loaded_encoders = calibrated_artifacts["encoders"]
loaded_destination_encoder = calibrated_artifacts["destination_encoder"]
loaded_feature_columns = calibrated_artifacts["feature_columns"]
loaded_threshold = calibrated_artifacts["threshold"]

print("Model loaded:", type(calibrated_model).__name__)
print("Questionnaire features:", len(loaded_feature_columns))
print("Decision threshold:", loaded_threshold)
print("Destination count:", len(destination_names))

Model loaded: CalibratedClassifierCV
Questionnaire features: 9
Decision threshold: 0.55
Destination count: 1486


In [55]:
# Test the calibrated model on the existing test data

calibrated_test_probabilities = calibrated_model.predict_proba(X_test)[:, 1]

print("First 5 calibrated predictions:")
print(calibrated_test_probabilities[:5].round(4))

print("\nProbability range:")
print("Minimum:", round(calibrated_test_probabilities.min(), 4))
print("Maximum:", round(calibrated_test_probabilities.max(), 4))

print("\nAll probabilities valid:",
      bool(((calibrated_test_probabilities >= 0) &
            (calibrated_test_probabilities <= 1)).all()))

First 5 calibrated predictions:
[0.4021 0.2685 0.2954 0.3391 0.3691]

Probability range:
Minimum: 0.0245
Maximum: 0.864

All probabilities valid: True


In [56]:
# Compare recommendations for two different travelers

traveler_1 = df.iloc[0]
traveler_2 = df.iloc[100]

answers_1 = {
    column: traveler_1[column]
    for column in loaded_feature_columns
}

answers_2 = {
    column: traveler_2[column]
    for column in loaded_feature_columns
}

recommendations_1 = recommend_destinations(answers_1, top_n=5)
recommendations_2 = recommend_destinations(answers_2, top_n=5)

print("TRAVELER 1")
print(recommendations_1[["destination", "acceptance_probability"]].to_string(index=False))

print("\nTRAVELER 2")
print(recommendations_2[["destination", "acceptance_probability"]].to_string(index=False))

set_1 = set(recommendations_1["destination_id"])
set_2 = set(recommendations_2["destination_id"])

print("\nDestinations shared:", len(set_1 & set_2), "out of 5")

TRAVELER 1
            destination  acceptance_probability
           Kyoto, Japan                   73.66
          Amalfi, Italy                   71.69
           Nikko, Japan                   68.97
        Vernazza, Italy                   67.96
Taormina, Sicily, Italy                   67.57

TRAVELER 2
            destination  acceptance_probability
           Kyoto, Japan                   74.96
          Amalfi, Italy                   73.05
           Nikko, Japan                   70.39
        Vernazza, Italy                   69.44
Taormina, Sicily, Italy                   69.08

Destinations shared: 5 out of 5


In [57]:
# Compare questionnaire answers for the two travelers

comparison = pd.DataFrame({
    "feature": loaded_feature_columns,
    "traveler_1": [
        str(traveler_1[column])
        for column in loaded_feature_columns
    ],
    "traveler_2": [
        str(traveler_2[column])
        for column in loaded_feature_columns
    ]
})

comparison["different"] = (
    comparison["traveler_1"] != comparison["traveler_2"]
)

print(comparison.to_string(index=False))
print(
    "\nDifferent answers:",
    int(comparison["different"].sum()),
    "out of",
    len(loaded_feature_columns)
)

feature           traveler_1                traveler_2  different
 form_a           ['0', '1']                     ['1']       True
 form_b                ['1']                     ['2']       True
 form_c                ['2']                     ['2']      False
 form_f ['0', '1', '2', '7']           ['0', '1', '2']       True
 form_g      ['0', '2', '4'] ['1', '2', '4', '6', '7']       True
 form_h                ['1']                     ['1']      False
 form_i                ['1']                     ['1']      False
 form_j                ['1']                     ['2']       True
 form_k                ['2']                     ['0']       True

Different answers: 6 out of 9


In [58]:
for i, cell in enumerate(In[1:] if False else []):
    pass

In [59]:
for column in loaded_feature_columns:
    encoder = loaded_encoders[column]
    print(f"{column}: {len(encoder.classes_)} categories")

print("\nDestination categories:",
      len(loaded_destination_encoder.categories_[0]))

print("Current total features:", X.shape[1])

form_a: 5 categories
form_b: 5 categories
form_c: 5 categories
form_f: 9 categories
form_g: 9 categories
form_h: 4 categories
form_i: 4 categories
form_j: 4 categories
form_k: 4 categories

Destination categories: 1452
Current total features: 1501


In [60]:
print("Feature matrix shape:", X.shape)
print("Feature matrix format:", X.format)
print("Non-zero values:", X.nnz)
print(f"Sparse data memory: {X.data.nbytes / (1024**2):.2f} MB")
print(f"Index memory: {(X.indices.nbytes + X.indptr.nbytes) / (1024**2):.2f} MB")

Feature matrix shape: (787138, 1501)
Feature matrix format: csr
Non-zero values: 12924463
Sparse data memory: 49.30 MB
Index memory: 52.31 MB


In [61]:
from scipy.sparse import csr_matrix, hstack
import numpy as np

# Take a small sample from the existing training split
rng = np.random.default_rng(42)
sample_size = 20_000

sample_positions = rng.choice(
    len(train_idx),
    size=sample_size,
    replace=False
)

X_sample = X_train[sample_positions]
y_sample = y_train[sample_positions]

print("Sample feature matrix:", X_sample.shape)
print("Sample labels:", y_sample.shape)
print("Accepted examples:", int(y_sample.sum()))
print("Rejected examples:", int((y_sample == 0).sum()))

Sample feature matrix: (20000, 1501)
Sample labels: (20000,)
Accepted examples: 7254
Rejected examples: 12746


In [62]:
sample_traveler = df.iloc[0]

traveler_answers = {
    column: sample_traveler[column]
    for column in loaded_feature_columns
}

recommendations = recommend_destinations(
    traveler_answers,
    top_n=5
)

print(recommendations.to_string(index=False))

 destination_id             destination  acceptance_probability
            471            Kyoto, Japan                   73.66
            433           Amalfi, Italy                   71.69
            473            Nikko, Japan                   68.97
            459         Vernazza, Italy                   67.96
            432 Taormina, Sicily, Italy                   67.57


In [63]:
from scipy.sparse import csr_matrix, hstack
import numpy as np

rng = np.random.default_rng(42)

sample_size = 20_000

sample_positions = rng.choice(
    len(train_idx),
    size=sample_size,
    replace=False
)

X_sample = X_train[sample_positions]
y_sample = y_train[sample_positions]

print("Sample feature matrix:", X_sample.shape)
print("Sample labels:", y_sample.shape)
print("Accepted:", int(y_sample.sum()))
print("Rejected:", int((y_sample == 0).sum()))

Sample feature matrix: (20000, 1501)
Sample labels: (20000,)
Accepted: 7254
Rejected: 12746


In [64]:
questionnaire_feature_count = sum(
    len(loaded_encoders[column].classes_)
    for column in loaded_feature_columns
)

X_questionnaire_sample = X_sample[:, :questionnaire_feature_count]

X_destination_sample = X_sample[:, questionnaire_feature_count:]

print("Questionnaire features:", X_questionnaire_sample.shape)
print("Destination features:", X_destination_sample.shape)

Questionnaire features: (20000, 49)
Destination features: (20000, 1452)


In [65]:
from scipy.sparse import csr_matrix

# Find active questionnaire and destination features
interaction_rows = []
interaction_cols = []

questionnaire_coo = X_questionnaire_sample.tocsr()
destination_coo = X_destination_sample.tocsr()

for row in range(X_sample.shape[0]):

    q_features = questionnaire_coo.indices[
        questionnaire_coo.indptr[row]:questionnaire_coo.indptr[row + 1]
    ]

    d_features = destination_coo.indices[
        destination_coo.indptr[row]:destination_coo.indptr[row + 1]
    ]

    for q in q_features:
        for d in d_features:
            interaction_rows.append(row)
            interaction_cols.append(
                q * X_destination_sample.shape[1] + d
            )

interaction_features = csr_matrix(
    (
        np.ones(len(interaction_rows), dtype=np.float32),
        (interaction_rows, interaction_cols)
    ),
    shape=(
        X_sample.shape[0],
        questionnaire_feature_count * X_destination_sample.shape[1]
    )
)

print("Interaction matrix:", interaction_features.shape)
print("Non-zero values:", interaction_features.nnz)
print(f"Memory: {interaction_features.data.nbytes / (1024**2):.2f} MB")

Interaction matrix: (20000, 71148)
Non-zero values: 307263
Memory: 1.17 MB


In [66]:
from scipy.sparse import hstack

X_sample_enhanced = hstack(
    [X_sample, interaction_features],
    format="csr",
    dtype=np.float32
)

print("Enhanced feature matrix:", X_sample_enhanced.shape)

Enhanced feature matrix: (20000, 72649)


In [67]:
from sklearn.model_selection import GroupShuffleSplit

sample_groups = groups[train_idx][sample_positions]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

sample_train_idx, sample_val_idx = next(
    splitter.split(X_sample_enhanced, y_sample, groups=sample_groups)
)

X_enhanced_train = X_sample_enhanced[sample_train_idx]
X_enhanced_val = X_sample_enhanced[sample_val_idx]

y_enhanced_train = y_sample[sample_train_idx]
y_enhanced_val = y_sample[sample_val_idx]

print("Training shape:", X_enhanced_train.shape)
print("Validation shape:", X_enhanced_val.shape)
print("Training travelers:", len(set(sample_groups[sample_train_idx])))
print("Validation travelers:", len(set(sample_groups[sample_val_idx])))

Training shape: (16006, 72649)
Validation shape: (3994, 72649)
Training travelers: 12024
Validation travelers: 3006


In [68]:
from sklearn.linear_model import LogisticRegression
import time

interaction_model = LogisticRegression(
    max_iter=200,
    solver="saga",
    class_weight="balanced",
    random_state=42
)

start_time = time.time()

interaction_model.fit(
    X_enhanced_train,
    y_enhanced_train
)

end_time = time.time()

print("Model training completed!")
print("Training time:", round(end_time - start_time, 2), "seconds")

Model training completed!
Training time: 1.9 seconds


In [69]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

val_predictions = interaction_model.predict(X_enhanced_val)

val_probabilities = interaction_model.predict_proba(
    X_enhanced_val
)[:, 1]

print("Accuracy:", round(accuracy_score(y_enhanced_val, val_predictions), 4))
print("Precision:", round(precision_score(y_enhanced_val, val_predictions), 4))
print("Recall:", round(recall_score(y_enhanced_val, val_predictions), 4))
print("F1 Score:", round(f1_score(y_enhanced_val, val_predictions), 4))
print("ROC-AUC:", round(roc_auc_score(y_enhanced_val, val_probabilities), 4))
print("Average Precision:", round(average_precision_score(y_enhanced_val, val_probabilities), 4))

Accuracy: 0.6462
Precision: 0.5128
Recall: 0.5399
F1 Score: 0.526
ROC-AUC: 0.6713
Average Precision: 0.5216


In [70]:
original_val_X = X_sample[sample_val_idx]
original_val_y = y_sample[sample_val_idx]

original_predictions = model.predict(original_val_X)

original_probabilities = model.predict_proba(
    original_val_X
)[:, 1]

print("Original Model Results")
print("Accuracy:", round(accuracy_score(original_val_y, original_predictions), 4))
print("Precision:", round(precision_score(original_val_y, original_predictions), 4))
print("Recall:", round(recall_score(original_val_y, original_predictions), 4))
print("F1 Score:", round(f1_score(original_val_y, original_predictions), 4))
print("ROC-AUC:", round(roc_auc_score(original_val_y, original_probabilities), 4))
print("Average Precision:", round(average_precision_score(original_val_y, original_probabilities), 4))

Original Model Results
Accuracy: 0.6592
Precision: 0.5222
Recall: 0.7376
F1 Score: 0.6115
ROC-AUC: 0.733
Average Precision: 0.6009


In [71]:
from pathlib import Path

data_folder = Path("data")

for file in data_folder.iterdir():
    print(file.name)

clean_training_data.csv
destination_features.csv
destination_ids.txt
external
matched_attractions.csv
Stravl_Travel_Preference_Data.csv
training_data.csv


In [72]:
original_df = pd.read_csv(
    "data/Stravl_Travel_Preference_Data.csv"
)

print("Dataset shape:", original_df.shape)

print("\nColumn names:")
for column in original_df.columns:
    print(column)

Dataset shape: (80301, 38)

Column names:
id
form_a
form_b
form_c
form_f
form_g
form_h
form_i
form_j
form_k
form_r
form_rr
yes_swipes
no_swipes
maybe_swipes
Model
Retrieval
DynaMatch
Rating_0
Rating_1
Rating_2
Rating_3
Rating_4
Rating_5
Rating_6
Rating_7
Rating_8
Rating_9
Rec_0
Rec_1
Rec_2
Rec_3
Rec_4
Rec_5
Rec_6
Rec_7
Rec_8
Rec_9


In [73]:
for index, column in enumerate(original_df.columns, start=1):
    print(index, ":", column)

1 : id
2 : form_a
3 : form_b
4 : form_c
5 : form_f
6 : form_g
7 : form_h
8 : form_i
9 : form_j
10 : form_k
11 : form_r
12 : form_rr
13 : yes_swipes
14 : no_swipes
15 : maybe_swipes
16 : Model
17 : Retrieval
18 : DynaMatch
19 : Rating_0
20 : Rating_1
21 : Rating_2
22 : Rating_3
23 : Rating_4
24 : Rating_5
25 : Rating_6
26 : Rating_7
27 : Rating_8
28 : Rating_9
29 : Rec_0
30 : Rec_1
31 : Rec_2
32 : Rec_3
33 : Rec_4
34 : Rec_5
35 : Rec_6
36 : Rec_7
37 : Rec_8
38 : Rec_9


In [74]:
rating_rec_columns = [
    column for column in original_df.columns
    if column.startswith("Rating_") or column.startswith("Rec_")
]

print(original_df[rating_rec_columns].head(10).to_string())

   Rating_0  Rating_1  Rating_2  Rating_3  Rating_4  Rating_5  Rating_6  Rating_7  Rating_8  Rating_9 Rec_0 Rec_1 Rec_2 Rec_3 Rec_4 Rec_5 Rec_6 Rec_7 Rec_8 Rec_9
0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
1      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
2      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
3      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
4      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0      -1.0    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
5      -1.0      -1.0      -

In [75]:
for column in rating_rec_columns:
    valid_count = (original_df[column] != -1).sum()

    print(
        column,
        "| Valid values:", valid_count,
        "| Missing indicator (-1):",
        (original_df[column] == -1).sum()
    )

Rating_0 | Valid values: 17611 | Missing indicator (-1): 62690
Rating_1 | Valid values: 17855 | Missing indicator (-1): 62446
Rating_2 | Valid values: 17893 | Missing indicator (-1): 62408
Rating_3 | Valid values: 17900 | Missing indicator (-1): 62401
Rating_4 | Valid values: 17913 | Missing indicator (-1): 62388
Rating_5 | Valid values: 17856 | Missing indicator (-1): 62445
Rating_6 | Valid values: 17985 | Missing indicator (-1): 62316
Rating_7 | Valid values: 17981 | Missing indicator (-1): 62320
Rating_8 | Valid values: 17982 | Missing indicator (-1): 62319
Rating_9 | Valid values: 17976 | Missing indicator (-1): 62325
Rec_0 | Valid values: 80301 | Missing indicator (-1): 0
Rec_1 | Valid values: 80301 | Missing indicator (-1): 0
Rec_2 | Valid values: 80301 | Missing indicator (-1): 0
Rec_3 | Valid values: 80301 | Missing indicator (-1): 0
Rec_4 | Valid values: 80301 | Missing indicator (-1): 0
Rec_5 | Valid values: 80301 | Missing indicator (-1): 0
Rec_6 | Valid values: 80301 | Miss

In [76]:
print("Recommendation data types:")
print(original_df[[f"Rec_{i}" for i in range(10)]].dtypes)

print("\nSample recommendation values:")
print(original_df[[f"Rec_{i}" for i in range(10)]].head(10).to_string())

print("\nUnique values in Rec_0:")
print(original_df["Rec_0"].nunique())
print(original_df["Rec_0"].unique()[:20])

Recommendation data types:
Rec_0    object
Rec_1    object
Rec_2    object
Rec_3    object
Rec_4    object
Rec_5    object
Rec_6    object
Rec_7    object
Rec_8    object
Rec_9    object
dtype: object

Sample recommendation values:
  Rec_0 Rec_1 Rec_2 Rec_3 Rec_4 Rec_5 Rec_6 Rec_7 Rec_8 Rec_9
0    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
1    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
2    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
3    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
4    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
5    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
6    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
7    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
8    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
9    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1

Unique values in Rec_0:
1285
['-1' 'Zaragoza, Spain' 'Sao Paulo, Brazil' 'Ghent, Belg

In [77]:
rec_columns = [f"Rec_{i}" for i in range(10)]

valid_recommendations = original_df[
    (original_df[rec_columns] != "-1").any(axis=1)
]

print("Travelers with recommendations:", len(valid_recommendations))

print("\nSample recommendation records:")

print(
    valid_recommendations[
        ["id"] + rec_columns + [f"Rating_{i}" for i in range(10)]
    ].head(5).to_string(index=False)
)

Travelers with recommendations: 18047

Sample recommendation records:
                              id             Rec_0            Rec_1                                     Rec_2            Rec_3               Rec_4 Rec_5 Rec_6 Rec_7 Rec_8 Rec_9  Rating_0  Rating_1  Rating_2  Rating_3  Rating_4  Rating_5  Rating_6  Rating_7  Rating_8  Rating_9
738932fcd28011ed86bd1238d4348226   Zaragoza, Spain     Cadiz, Spain                           Bremen, Germany   Warsaw, Poland       Madrid, Spain   NaN   NaN   NaN   NaN   NaN       0.0       0.0       0.0       0.0       0.0       NaN       NaN       NaN       NaN       NaN
a0c91202d28311eda45116bdf5dc994c Sao Paulo, Brazil Bogota, Colombia                            Santos, Brazil Brasilia, Brazil      Manaus, Brazil   NaN   NaN   NaN   NaN   NaN       0.0       0.0       0.0       0.0       0.0       NaN       NaN       NaN       NaN       NaN
96aa1a02ba1211edad17acde48001122    Ghent, Belgium Košice, Slovakia Colorado Springs, Colorado, Uni

In [78]:
rating_columns = [f"Rating_{i}" for i in range(10)]

all_ratings = original_df[rating_columns].values.flatten()

rating_counts = pd.Series(all_ratings).value_counts(dropna=False).sort_index()

print("Rating distribution:")
print(rating_counts)

Rating distribution:
-1.0    624058
 0.0     77074
 1.0      1343
 NaN    100535
Name: count, dtype: int64


In [79]:
rating_summary = []

for i in range(10):
    rec_col = f"Rec_{i}"
    rating_col = f"Rating_{i}"

    valid = (
        original_df[rec_col].notna()
        & (original_df[rec_col] != "-1")
        & original_df[rating_col].isin([0, 1])
    )

    rating_summary.append({
        "Recommendation": rec_col,
        "Rating": rating_col,
        "Valid feedback": valid.sum(),
        "Rating 0": (valid & (original_df[rating_col] == 0)).sum(),
        "Rating 1": (valid & (original_df[rating_col] == 1)).sum()
    })

pd.DataFrame(rating_summary)

,Recommendation,Rating,Valid feedback,Rating 0,Rating 1
0,Rec_0,Rating_0,12928,12581,347
1,Rec_1,Rating_1,13172,12982,190
2,Rec_2,Rating_2,13210,13047,163
3,Rec_3,Rating_3,13217,13081,136
4,Rec_4,Rating_4,13230,13094,136
5,Rec_5,Rating_5,2431,2317,114
6,Rec_6,Rating_6,2560,2489,71
7,Rec_7,Rating_7,2556,2485,71
8,Rec_8,Rating_8,2557,2497,60
9,Rec_9,Rating_9,2551,2496,55


In [80]:
recommendation_records = valid_recommendations[
    ["id"] + rec_columns
].copy()

print("Unique traveler IDs:", recommendation_records["id"].nunique())

print("\nFirst 10 recommendation records:")
print(
    recommendation_records.head(10).to_string(index=False)
)

Unique traveler IDs: 18047

First 10 recommendation records:
                              id                              Rec_0                                            Rec_1                                     Rec_2                                   Rec_3                              Rec_4                                  Rec_5                           Rec_6                     Rec_7             Rec_8                             Rec_9
738932fcd28011ed86bd1238d4348226                    Zaragoza, Spain                                     Cadiz, Spain                           Bremen, Germany                          Warsaw, Poland                      Madrid, Spain                                    NaN                             NaN                       NaN               NaN                               NaN
a0c91202d28311eda45116bdf5dc994c                  Sao Paulo, Brazil                                 Bogota, Colombia                            Santos, Brazil               

In [81]:
rec_columns = [f"Rec_{i}" for i in range(10)]

valid_rec_mask = (
    original_df[rec_columns].notna()
    & original_df[rec_columns].ne("-1")
).any(axis=1)

actual_recommendations = original_df[valid_rec_mask].copy()

print("Total travelers:", len(original_df))
print("Travelers with actual recommendations:", len(actual_recommendations))
print("Unique traveler IDs:", actual_recommendations["id"].nunique())

print("\nSample:")
print(
    actual_recommendations[["id"] + rec_columns]
    .head(5)
    .to_string(index=False)
)

Total travelers: 80301
Travelers with actual recommendations: 13358
Unique traveler IDs: 13358

Sample:
                              id             Rec_0            Rec_1                                     Rec_2              Rec_3               Rec_4 Rec_5 Rec_6 Rec_7 Rec_8 Rec_9
738932fcd28011ed86bd1238d4348226   Zaragoza, Spain     Cadiz, Spain                           Bremen, Germany     Warsaw, Poland       Madrid, Spain   NaN   NaN   NaN   NaN   NaN
a0c91202d28311eda45116bdf5dc994c Sao Paulo, Brazil Bogota, Colombia                            Santos, Brazil   Brasilia, Brazil      Manaus, Brazil   NaN   NaN   NaN   NaN   NaN
96aa1a02ba1211edad17acde48001122    Ghent, Belgium Košice, Slovakia Colorado Springs, Colorado, United States      Malmö, Sweden Luzern, Switzerland   NaN   NaN   NaN   NaN   NaN
b68f5a24d28311edadc00e96bc2eb5a0    Thasos, Greece     Begur, Spain                           Figueres, Spain Hallstatt, Austria      Kavala, Greece   NaN   NaN   NaN   NaN   NaN
f

In [82]:
recommendation_travelers = set(actual_recommendations["id"])

interaction_travelers = set(df["traveler_id"])

common_travelers = recommendation_travelers.intersection(
    interaction_travelers
)

print("Travelers with recommendations:", len(recommendation_travelers))
print("Travelers with swipe interactions:", len(interaction_travelers))
print("Travelers present in both datasets:", len(common_travelers))

Travelers with recommendations: 13358
Travelers with swipe interactions: 60886
Travelers present in both datasets: 13283


In [83]:
print("Destination mapping examples:")

for i, name in enumerate(destination_names[:10]):
    print(i, ":", name)

Destination mapping examples:
0 : Butrint, Albania
1 : Tirana, Albania
2 : Algiers, Algeria
3 : Oran, Algeria
4 : Andorra la Vella, Andorra
5 : Jolly Harbour, Antigua and Barbuda
6 : St John's, Antigua and Barbuda
7 : Buenos Aires, Argentina
8 : Córdoba, Argentina
9 : Mar del Plata, Argentina


In [84]:
destination_to_id = {
    name: destination_id
    for destination_id, name in enumerate(destination_names)
}

print("Total destinations:", len(destination_to_id))

# Test the mapping
sample_destination = "Kyoto, Japan"

print(
    sample_destination,
    "->",
    destination_to_id.get(sample_destination, "Not found")
)

Total destinations: 1486
Kyoto, Japan -> 471


In [85]:
recommendation_id_columns = []

for i in range(10):
    rec_col = f"Rec_{i}"
    id_col = f"Rec_ID_{i}"

    actual_recommendations[id_col] = (
        actual_recommendations[rec_col]
        .map(destination_to_id)
    )

    recommendation_id_columns.append(id_col)

print(
    actual_recommendations[
        ["id"] + rec_columns[:5] + recommendation_id_columns[:5]
    ].head(10).to_string(index=False)
)

                              id                              Rec_0                                            Rec_1                                     Rec_2                                   Rec_3                              Rec_4  Rec_ID_0  Rec_ID_1  Rec_ID_2  Rec_ID_3  Rec_ID_4
738932fcd28011ed86bd1238d4348226                    Zaragoza, Spain                                     Cadiz, Spain                           Bremen, Germany                          Warsaw, Poland                      Madrid, Spain     759.0     738.0     315.0     648.0     745.0
a0c91202d28311eda45116bdf5dc994c                  Sao Paulo, Brazil                                 Bogota, Colombia                            Santos, Brazil                        Brasilia, Brazil                     Manaus, Brazil     114.0     168.0     113.0     100.0     107.0
96aa1a02ba1211edad17acde48001122                     Ghent, Belgium                                 Košice, Slovakia Colorado Springs, Colorado, Uni

In [86]:
total_recommendations = actual_recommendations[rec_columns].notna().sum().sum()

matched_recommendations = (
    actual_recommendations[recommendation_id_columns]
    .notna()
    .sum()
    .sum()
)

unmatched_recommendations = (
    total_recommendations - matched_recommendations
)

print("Total recommendation entries:", total_recommendations)
print("Successfully matched:", matched_recommendations)
print("Unmatched:", unmatched_recommendations)

print(
    "Match percentage:",
    round(matched_recommendations / total_recommendations * 100, 2),
    "%"
)

Total recommendation entries: 79857
Successfully matched: 79850
Unmatched: 7
Match percentage: 99.99 %


In [87]:
unmatched_destinations = set()

for column in rec_columns:
    unmatched_mask = (
        actual_recommendations[column].notna()
        & actual_recommendations[column].ne("-1")
        & actual_recommendations[column].map(destination_to_id).isna()
    )

    unmatched_destinations.update(
        actual_recommendations.loc[unmatched_mask, column].tolist()
    )

print("Unmatched destination names:")
for destination in unmatched_destinations:
    print(destination)

Unmatched destination names:
57
1454
55
999
['Monaco, Monaco', 'Clearwater, Florida, United States', 'Sintra, Portugal', 'Birmingham, Alabama, United States', 'Tulsa, Oklahoma, United States']
189


In [88]:
for column in rec_columns:
    numeric_mask = actual_recommendations[column].astype(str).str.fullmatch(r"\d+")

    if numeric_mask.any():
        print(f"\n{column}:")
        print(actual_recommendations.loc[numeric_mask, column].unique())


Rec_0:
['189']

Rec_1:
['55']

Rec_2:
['57']

Rec_3:
['1454']

Rec_4:
['999']


In [89]:
numeric_ids = [189, 55, 57, 1454, 999]

for destination_id in numeric_ids:
    print(destination_id, ":", destination_names[destination_id])

189 : Split, Croatia
55 : Nassau, Bahamas
57 : Bridgetown, Barbados
1454 : Madison, Wisconsin, United States
999 : Clearwater, Florida, United States


In [90]:
def convert_destination(value):
    if pd.isna(value) or str(value) == "-1":
        return np.nan

    value = str(value).strip()

    # If the value is already a numeric destination ID
    if value.isdigit():
        destination_id = int(value)

        if 0 <= destination_id < len(destination_names):
            return destination_id

    # If the value is a destination name
    return destination_to_id.get(value, np.nan)


for i in range(10):
    rec_col = f"Rec_{i}"
    id_col = f"Rec_ID_{i}"

    actual_recommendations[id_col] = (
        actual_recommendations[rec_col].apply(convert_destination)
    )

In [91]:
matched_recommendations = (
    actual_recommendations[recommendation_id_columns].notna().sum().sum()
)

print("Matched recommendations:", matched_recommendations)

Matched recommendations: 79855


In [92]:
unmatched_destinations = set()

for column in rec_columns:
    unmatched_mask = (
        actual_recommendations[column].notna()
        & actual_recommendations[column].ne("-1")
        & actual_recommendations[f"Rec_ID_{column.split('_')[1]}"].isna()
    )

    unmatched_destinations.update(
        actual_recommendations.loc[unmatched_mask, column].tolist()
    )

print("Remaining unmatched destinations:")
print(unmatched_destinations)

Remaining unmatched destinations:
{"['Monaco, Monaco', 'Clearwater, Florida, United States', 'Sintra, Portugal', 'Birmingham, Alabama, United States', 'Tulsa, Oklahoma, United States']"}


In [93]:
for destination in unmatched_destinations:
    count = sum(
        (actual_recommendations[column] == destination).sum()
        for column in rec_columns
    )

    print(destination, ":", count)

['Monaco, Monaco', 'Clearwater, Florida, United States', 'Sintra, Portugal', 'Birmingham, Alabama, United States', 'Tulsa, Oklahoma, United States'] : 1


In [94]:
import ast

unmatched_list = ast.literal_eval(next(iter(unmatched_destinations)))

for destination in unmatched_list:
    count = sum(
        (actual_recommendations[column] == destination).sum()
        for column in rec_columns
    )

    print(destination, ":", count)

Monaco, Monaco : 179
Clearwater, Florida, United States : 61
Sintra, Portugal : 106
Birmingham, Alabama, United States : 61
Tulsa, Oklahoma, United States : 38


In [95]:
for destination in unmatched_list:
    if destination not in destination_to_id:
        new_id = len(destination_names)

        destination_names.append(destination)
        destination_to_id[destination] = new_id

print("Updated destination count:", len(destination_names))

Updated destination count: 1486


In [96]:
print("Missing destination list:")

for destination in unmatched_list:
    print(
        repr(destination),
        "| Already in mapping:",
        destination in destination_to_id
    )

print("\nDestination mapping size:", len(destination_to_id))

Missing destination list:
'Monaco, Monaco' | Already in mapping: True
'Clearwater, Florida, United States' | Already in mapping: True
'Sintra, Portugal' | Already in mapping: True
'Birmingham, Alabama, United States' | Already in mapping: True
'Tulsa, Oklahoma, United States' | Already in mapping: True

Destination mapping size: 1486


In [97]:
for destination in unmatched_list:
    print(destination, ":", destination_to_id[destination])

Monaco, Monaco : 541
Clearwater, Florida, United States : 999
Sintra, Portugal : 665
Birmingham, Alabama, United States : 865
Tulsa, Oklahoma, United States : 1307


In [98]:
for i in range(10):
    rec_col = f"Rec_{i}"
    id_col = f"Rec_ID_{i}"

    actual_recommendations[id_col] = (
        actual_recommendations[rec_col].apply(convert_destination)
    )

matched_recommendations = (
    actual_recommendations[recommendation_id_columns].notna().sum().sum()
)

total_recommendations = (
    actual_recommendations[rec_columns].notna()
    & actual_recommendations[rec_columns].ne("-1")
).sum().sum()

print("Total recommendations:", total_recommendations)
print("Matched recommendations:", matched_recommendations)
print("Unmatched recommendations:", total_recommendations - matched_recommendations)

Total recommendations: 79856
Matched recommendations: 79855
Unmatched recommendations: 1


In [99]:
for column in rec_columns:
    id_col = f"Rec_ID_{column.split('_')[1]}"

    unmatched_mask = (
        actual_recommendations[column].notna()
        & actual_recommendations[column].ne("-1")
        & actual_recommendations[id_col].isna()
    )

    if unmatched_mask.any():
        print(column)
        print(actual_recommendations.loc[unmatched_mask, column].value_counts())

Rec_0
Rec_0
['Monaco, Monaco', 'Clearwater, Florida, United States', 'Sintra, Portugal', 'Birmingham, Alabama, United States', 'Tulsa, Oklahoma, United States']    1
Name: count, dtype: int64


In [100]:
bad_value = str(unmatched_list)

bad_rows = actual_recommendations[
    actual_recommendations["Rec_0"].astype(str) == bad_value
]

print(bad_rows[["id", "Rec_0", "Rating_0"]])

                                     id  \
45812  de8b770b8c3611edb10748e7dac4d998   

                                                   Rec_0  Rating_0  
45812  ['Monaco, Monaco', 'Clearwater, Florida, Unite...      -1.0  


In [101]:
bad_value = actual_recommendations.loc[45812, "Rec_0"]

actual_recommendations.loc[
    actual_recommendations["Rec_0"] == bad_value,
    "Rec_ID_0"
] = np.nan

print("Corrected malformed recommendation.")

Corrected malformed recommendation.


In [102]:
print(
    actual_recommendations[recommendation_id_columns]
    .notna()
    .sum()
    .sum()
)

79855


In [103]:
rating_columns = [f"Rating_{i}" for i in range(10)]

rating_counts = (
    original_df[rating_columns]
    .stack()
    .value_counts()
    .sort_index()
)

print(rating_counts)

-1.0    624058
 0.0     77074
 1.0      1343
Name: count, dtype: int64


In [104]:
for i in range(10):
    rec_col = f"Rec_{i}"
    rating_col = f"Rating_{i}"

    valid_rows = original_df[
        (original_df[rec_col].notna()) &
        (original_df[rec_col].astype(str) != "-1") &
        (original_df[rating_col].isin([0, 1]))
    ]

    print(f"\nRecommendation {i}")

    print(
        valid_rows[[rec_col, rating_col]].head(5)
    )


Recommendation 0
                 Rec_0  Rating_0
208    Zaragoza, Spain       0.0
253  Sao Paulo, Brazil       0.0
691     Ghent, Belgium       0.0
787     Thasos, Greece       0.0
854    Yangon, Myanmar       0.0

Recommendation 1
                Rec_1  Rating_1
208      Cadiz, Spain       0.0
253  Bogota, Colombia       0.0
691  Košice, Slovakia       0.0
787      Begur, Spain       0.0
854   Beirut, Lebanon       0.0

Recommendation 2
                                         Rec_2  Rating_2
208                            Bremen, Germany       0.0
253                             Santos, Brazil       0.0
691  Colorado Springs, Colorado, United States       0.0
787                            Figueres, Spain       0.0
854                           Nazareth, Israel       0.0

Recommendation 3
                  Rec_3  Rating_3
208      Warsaw, Poland       0.0
253    Brasilia, Brazil       0.0
691       Malmö, Sweden       0.0
787  Hallstatt, Austria       0.0
854  Neve Zohar, Israel   

In [105]:
for i in range(10):
    rating_col = f"Rating_{i}"

    print(f"\nRating {i}:")
    print(original_df[rating_col].value_counts())


Rating 0:
Rating_0
-1.0    62690
 0.0    12582
 1.0      347
Name: count, dtype: int64

Rating 1:
Rating_1
-1.0    62446
 0.0    12983
 1.0      190
Name: count, dtype: int64

Rating 2:
Rating_2
-1.0    62408
 0.0    13048
 1.0      163
Name: count, dtype: int64

Rating 3:
Rating_3
-1.0    62401
 0.0    13082
 1.0      136
Name: count, dtype: int64

Rating 4:
Rating_4
-1.0    62388
 0.0    13095
 1.0      136
Name: count, dtype: int64

Rating 5:
Rating_5
-1.0    62445
 0.0     2317
 1.0      114
Name: count, dtype: int64

Rating 6:
Rating_6
-1.0    62316
 0.0     2489
 1.0       71
Name: count, dtype: int64

Rating 7:
Rating_7
-1.0    62320
 0.0     2485
 1.0       71
Name: count, dtype: int64

Rating 8:
Rating_8
-1.0    62319
 0.0     2497
 1.0       60
Name: count, dtype: int64

Rating 9:
Rating_9
-1.0    62325
 0.0     2496
 1.0       55
Name: count, dtype: int64


In [106]:
for i in range(10):
    rec_col = f"Rec_{i}"
    rating_col = f"Rating_{i}"

    liked = original_df[
        original_df[rating_col] == 1
    ][[rec_col, rating_col]]

    print(f"\nRecommendation {i} — Rating 1")
    print(liked.head(5))


Recommendation 0 — Rating 1
                                      Rec_0  Rating_0
8465                      Sao Paulo, Brazil       1.0
9040                         Cancun, Mexico       1.0
18493  Boston, Massachusetts, United States       1.0
22733                      Hiroshima, Japan       1.0
28049                       Larnaca, Cyprus       1.0

Recommendation 1 — Rating 1
                                       Rec_1  Rating_1
20484                     Ushuaia, Argentina       1.0
21833                     Zanzibar, Tanzania       1.0
42237         Kahului, Hawaii, United States       1.0
45141                     Ouarzazat, Morocco       1.0
46588  Poughkeepsie, New York, United States       1.0

Recommendation 2 — Rating 1
                                      Rec_2  Rating_2
49928                        Recife, Brazil       1.0
50331     Vaitape, French Polynesia, France       1.0
52344      Boulder, Colorado, United States       1.0
52387                     Perigueux, France

In [107]:
print(
    original_df[
        ["id", "yes_swipes", "no_swipes", "maybe_swipes"]
    ].head(10).to_string(index=False)
)

                              id                                     yes_swipes                                                                                                                                                                                                                                               no_swipes maybe_swipes
22871c2cb8ee11ec9a19fefb5d5b7ead [837, 779, 1315, 773, 621, 799, 794, 172, 190]                                                                                                                                                                                              [1256, 855, 1259, 551, 55, 257, 648, 234, 209, 1208, 1381]           []
89b7022ab8f011ec9a19fefb5d5b7ead                                     [246, 579]                                                                                                                                                                                                               [126, 945, 865, 330, 1237, 1264,

In [108]:
sample = original_df[original_df["Rating_0"] == 1].iloc[0]

print("Traveler ID:", sample["id"])
print("Recommended destination:", sample["Rec_0"])
print("Rating:", sample["Rating_0"])
print("Yes swipes:", sample["yes_swipes"])
print("No swipes:", sample["no_swipes"])
print("Maybe swipes:", sample["maybe_swipes"])

Traveler ID: 57d32976d28211edb342cef98186c1c7
Recommended destination: Sao Paulo, Brazil
Rating: 1.0
Yes swipes: [168, 13, 233, 631, 96, 232, 628]
No swipes: [630, 140, 141]
Maybe swipes: []


In [109]:
import ast

yes_list = ast.literal_eval(sample["yes_swipes"])
no_list = ast.literal_eval(sample["no_swipes"])
maybe_list = ast.literal_eval(sample["maybe_swipes"])

destination_id = destination_to_id["Sao Paulo, Brazil"]

print("Destination ID:", destination_id)
print("In yes_swipes:", destination_id in yes_list)
print("In no_swipes:", destination_id in no_list)
print("In maybe_swipes:", destination_id in maybe_list)

Destination ID: 114
In yes_swipes: False
In no_swipes: False
In maybe_swipes: False


In [110]:
sample_zero = original_df[
    (original_df["Rating_0"] == 0) &
    (original_df["Rec_0"].notna()) &
    (original_df["Rec_0"] != "-1")
].iloc[0]

destination_zero_id = destination_to_id[sample_zero["Rec_0"]]

yes_zero = ast.literal_eval(sample_zero["yes_swipes"])
no_zero = ast.literal_eval(sample_zero["no_swipes"])
maybe_zero = ast.literal_eval(sample_zero["maybe_swipes"])

print("Destination:", sample_zero["Rec_0"])
print("Destination ID:", destination_zero_id)
print("Rating:", sample_zero["Rating_0"])
print("In yes_swipes:", destination_zero_id in yes_zero)
print("In no_swipes:", destination_zero_id in no_zero)
print("In maybe_swipes:", destination_zero_id in maybe_zero)

Destination: Zaragoza, Spain
Destination ID: 759
Rating: 0.0
In yes_swipes: False
In no_swipes: False
In maybe_swipes: False


In [111]:
print(original_df[
    ["Model", "Retrieval", "DynaMatch"]
].head(10).to_string(index=False))

 Model  Retrieval  DynaMatch
    -1         -1         -1
    -1         -1         -1
    -1         -1         -1
    -1         -1         -1
    -1         -1         -1
    -1         -1         -1
    -1         -1         -1
    -1         -1         -1
    -1         -1         -1
    -1         -1         -1


In [112]:
metadata_columns = ["Model", "Retrieval", "DynaMatch"]

for column in metadata_columns:
    print(f"\n{column}:")
    print(original_df[column].value_counts(dropna=False).head(15))


Model:
Model
-1    63615
 1     2772
 2     2687
 4     2644
 5     2612
 0     2594
 3     2594
 6      782
 7        1
Name: count, dtype: int64

Retrieval:
Retrieval
-1    63615
 0    11648
 1     5038
Name: count, dtype: int64

DynaMatch:
DynaMatch
-1    63615
 0    14983
 2      852
 1      851
Name: count, dtype: int64


In [113]:
positive_metadata = original_df[
    original_df["Rating_0"] == 1
][["Rec_0", "Rating_0", "Model", "Retrieval", "DynaMatch"]]

print(positive_metadata.head(15).to_string(index=False))

                               Rec_0  Rating_0  Model  Retrieval  DynaMatch
                   Sao Paulo, Brazil       1.0      0          1          0
                      Cancun, Mexico       1.0      2          1          2
Boston, Massachusetts, United States       1.0      1          0          0
                    Hiroshima, Japan       1.0      6          0          0
                     Larnaca, Cyprus       1.0      2          1          0
             Bastia, Corsica, France       1.0      3          0          0
                    Hiroshima, Japan       1.0      2          1          0
             Bastia, Corsica, France       1.0      5          0          0
                      Cancun, Mexico       1.0      0          0          0
 Edinburgh, Scotland, United Kingdom       1.0      5          1          0
            Heraklion, Crete, Greece       1.0      6          1          0
                      Aegina, Greece       1.0      1          0          1
        Neum

In [114]:
for rating in [0, 1]:

    subset = original_df[original_df["Rating_0"] == rating]

    print(f"\nRating = {rating}")

    for column in ["Model", "Retrieval", "DynaMatch"]:
        print(f"\n{column}:")
        print(subset[column].value_counts())


Rating = 0

Model:
Model
 1    1954
 2    1907
 5    1845
 0    1828
 4    1814
 3    1780
-1     908
 6     545
 7       1
Name: count, dtype: int64

Retrieval:
Retrieval
 0    8178
 1    3496
-1     908
Name: count, dtype: int64

DynaMatch:
DynaMatch
 0    10579
-1      908
 1      577
 2      518
Name: count, dtype: int64

Rating = 1

Model:
Model
 4    66
 2    56
 3    48
 1    46
 0    43
 5    36
-1    30
 6    22
Name: count, dtype: int64

Retrieval:
Retrieval
 0    210
 1    107
-1     30
Name: count, dtype: int64

DynaMatch:
DynaMatch
 0    276
-1     30
 2     28
 1     13
Name: count, dtype: int64


In [115]:
for column in ["Model", "Retrieval", "DynaMatch"]:

    print(f"\n--- {column} ---")

    comparison = pd.crosstab(
        original_df["Rating_0"],
        original_df[column],
        normalize="index"
    ) * 100

    print(comparison.round(2))


--- Model ---
Model        -1      0      1      2      3      4      5     6     7
Rating_0                                                             
-1.0      99.37   0.09   0.10   0.10   0.09   0.10   0.13  0.01  0.00
 0.0       7.22  14.53  15.53  15.16  14.15  14.42  14.66  4.33  0.01
 1.0       8.65  12.39  13.26  16.14  13.83  19.02  10.37  6.34  0.00

--- Retrieval ---
Retrieval     -1      0      1
Rating_0                      
-1.0       99.37   0.42   0.21
 0.0        7.22  65.00  27.79
 1.0        8.65  60.52  30.84

--- DynaMatch ---
DynaMatch     -1      0     1     2
Rating_0                           
-1.0       99.37   0.58  0.02  0.03
 0.0        7.22  84.08  4.59  4.12
 1.0        8.65  79.54  3.75  8.07


In [116]:
rating_data = []

for i in range(10):

    rec_col = f"Rec_{i}"
    rating_col = f"Rating_{i}"

    valid = original_df[
        original_df[rec_col].notna()
        & original_df[rec_col].astype(str).ne("-1")
        & original_df[rating_col].isin([0, 1])
    ]

    rating_data.append({
        "Recommendation_Position": i,
        "Valid_Feedback": len(valid),
        "Rating_0": (valid[rating_col] == 0).sum(),
        "Rating_1": (valid[rating_col] == 1).sum()
    })

rating_summary = pd.DataFrame(rating_data)

print(rating_summary.to_string(index=False))
print("\nTotal valid feedback:", rating_summary["Valid_Feedback"].sum())

 Recommendation_Position  Valid_Feedback  Rating_0  Rating_1
                       0           12928     12581       347
                       1           13172     12982       190
                       2           13210     13047       163
                       3           13217     13081       136
                       4           13230     13094       136
                       5            2431      2317       114
                       6            2560      2489        71
                       7            2556      2485        71
                       8            2557      2497        60
                       9            2551      2496        55

Total valid feedback: 78412


In [117]:
valid_travelers = set()

for i in range(10):

    rec_col = f"Rec_{i}"
    rating_col = f"Rating_{i}"

    valid_mask = (
        original_df[rec_col].notna()
        & original_df[rec_col].astype(str).ne("-1")
        & original_df[rating_col].isin([0, 1])
    )

    valid_travelers.update(
        original_df.loc[valid_mask, "id"].tolist()
    )

print("Unique travelers with valid feedback:", len(valid_travelers))

Unique travelers with valid feedback: 13331


In [118]:
swipe_travelers = set(df["traveler_id"].unique())

common_travelers = valid_travelers.intersection(swipe_travelers)

print("Feedback travelers:", len(valid_travelers))
print("Swipe dataset travelers:", len(swipe_travelers))
print("Common travelers:", len(common_travelers))
print("Feedback-only travelers:", len(valid_travelers - swipe_travelers))

Feedback travelers: 13331
Swipe dataset travelers: 60886
Common travelers: 13256
Feedback-only travelers: 75


In [119]:
swipe_pairs = set(
    zip(df["traveler_id"], df["destination_id"])
)

feedback_pairs = set()

for i in range(10):

    rec_col = f"Rec_ID_{i}"
    rating_col = f"Rating_{i}"

    valid_mask = (
        actual_recommendations[rec_col].notna()
        & actual_recommendations[rating_col].isin([0, 1])
    )

    pairs = zip(
        actual_recommendations.loc[valid_mask, "id"],
        actual_recommendations.loc[valid_mask, rec_col].astype(int)
    )

    feedback_pairs.update(pairs)

common_pairs = swipe_pairs.intersection(feedback_pairs)

print("Swipe pairs:", len(swipe_pairs))
print("Feedback pairs:", len(feedback_pairs))
print("Common pairs:", len(common_pairs))

Swipe pairs: 787138
Feedback pairs: 76672
Common pairs: 918


In [120]:
overlap_examples = list(common_pairs)[:10]

overlap_df = pd.DataFrame(
    overlap_examples,
    columns=["traveler_id", "destination_id"]
)

overlap_df["destination"] = overlap_df["destination_id"].map(
    lambda x: destination_names[x]
)

print(overlap_df.to_string(index=False))

                     traveler_id  destination_id                            destination
053d423cbb8f11edb3da4eaa602dcf52             401                          Panaji, India
4f487b2abb9011edb3da4eaa602dcf52             855    Edinburgh, Scotland, United Kingdom
621c12d4bc5811ed9540baab856be9bb            1339 Rio Grande, Puerto Rico, United States
8d00ce3ad27f11eda45116bdf5dc994c             442                           Milan, Italy
117e02f6bfdc11eda6d7baf95b846ba9             439                       La Spezia, Italy
fca72e9ebbac11ed90b32eb4f2980715             366                         Zagori, Greece
7f5d23e4bbac11ed90b32eb4f2980715             558                     Maputo, Mozambique
59ee5876bacb11ed8a826af7a9aa8945             937 Los Angeles, California, United States
f221bef6bbb311edb7715225eb050bd9             470                       Hiroshima, Japan
4188d1f0bb0011edb15ee60116a18e06             439                       La Spezia, Italy


In [121]:
feedback_records = []

for i in range(10):
    rec_col = f"Rec_ID_{i}"
    rating_col = f"Rating_{i}"

    valid_mask = (
        actual_recommendations[rec_col].notna()
        & actual_recommendations[rating_col].isin([0, 1])
    )

    records = actual_recommendations.loc[
        valid_mask, ["id", rec_col, rating_col]
    ].copy()

    records.columns = ["traveler_id", "destination_id", "rating"]

    feedback_records.append(records)

feedback_df = pd.concat(feedback_records, ignore_index=True)

comparison_df = feedback_df.merge(
    df[["traveler_id", "destination_id", "accepted"]],
    on=["traveler_id", "destination_id"],
    how="inner"
)

print(pd.crosstab(
    comparison_df["rating"],
    comparison_df["accepted"]
))

accepted   0    1
rating           
0.0       58  914
1.0        0   37


In [122]:
rating_acceptance = (
    comparison_df.groupby("rating")["accepted"]
    .agg(["count", "sum", "mean"])
)

rating_acceptance["acceptance_percentage"] = (
    rating_acceptance["mean"] * 100
)

print(rating_acceptance.round(2))

        count  sum  mean  acceptance_percentage
rating                                         
0.0       972  914  0.94                  94.03
1.0        37   37  1.00                 100.00


In [123]:
print(destination_popularity.columns.tolist())
print(destination_popularity.head())

['destination_id', 'accepted_count', 'response_count', 'acceptance_rate', 'smoothed_rate']
     destination_id  accepted_count  response_count  acceptance_rate  \
471             471             557             748         0.744652   
433             433             256             352         0.727273   
459             459             241             340         0.708824   
473             473             227             321         0.707165   
354             354             305             446         0.683857   

     smoothed_rate  
471       0.734817  
433       0.707902  
459       0.689833  
473       0.687213  
354       0.670257  


In [124]:
rating_popularity = comparison_df.merge(
    destination_popularity[
        ["destination_id", "acceptance_rate", "response_count"]
    ],
    on="destination_id",
    how="left"
)

print(
    rating_popularity.groupby("rating")[
        ["acceptance_rate", "response_count"]
    ].mean().round(3)
)

        acceptance_rate  response_count
rating                                 
0.0               0.482         880.569
1.0               0.532         987.649


In [125]:
print(
    comparison_df.groupby("rating")["destination_id"]
    .nunique()
)

rating
0.0    368
1.0     31
Name: destination_id, dtype: int64


In [126]:
rating_consistency = (
    comparison_df.groupby("destination_id")["rating"]
    .nunique()
    .value_counts()
)

print(rating_consistency)

rating
1    353
2     23
Name: count, dtype: int64


In [127]:
mixed_destinations = (
    comparison_df.groupby("destination_id")["rating"]
    .nunique()
)

mixed_destinations = mixed_destinations[
    mixed_destinations == 2
].index

print(
    comparison_df[
        comparison_df["destination_id"].isin(mixed_destinations)
    ].groupby(["destination_id", "rating"])
    .size()
    .unstack(fill_value=0)
    .head(10)
)

rating          0.0  1.0
destination_id          
27.0             12    1
41.0              4    1
128.0             2    2
182.0            11    1
212.0            10    2
275.0            11    1
426.0             1    1
429.0             1    1
439.0            50    3
448.0             1    1


In [128]:
mixed_destination_names = pd.DataFrame({
    "destination_id": mixed_destinations
})

mixed_destination_names["destination_name"] = (
    mixed_destination_names["destination_id"]
    .astype(int)
    .map(lambda x: destination_names[x])
)

print(mixed_destination_names.head(10))

   destination_id                destination_name
0            27.0               Cairns, Australia
1            41.0              Innsbruck, Austria
2           128.0             Quebec City, Canada
3           182.0              Dubrovnik, Croatia
4           212.0          Prague, Czech Republic
5           275.0  Porto-Vecchio, Corsica, France
6           426.0       Cagliari, Sardinia, Italy
7           429.0          Catania, Sicily, Italy
8           439.0                La Spezia, Italy
9           448.0                   Pompei, Italy


In [129]:
print(
    comparison_df.groupby("traveler_id")["rating"]
    .nunique()
    .value_counts()
)

rating
1    679
2     11
Name: count, dtype: int64


In [130]:
mixed_rating_travelers = (
    comparison_df.groupby("traveler_id")["rating"]
    .nunique()
)

mixed_rating_travelers = mixed_rating_travelers[
    mixed_rating_travelers == 2
].index

print(
    comparison_df[
        comparison_df["traveler_id"].isin(mixed_rating_travelers)
    ][["traveler_id", "destination_id", "rating"]]
    .sort_values("traveler_id")
    .head(20)
)

                          traveler_id  destination_id  rating
986  45e2642db72311edae74cee2c33601bf           189.0     0.0
87   45e2642db72311edae74cee2c33601bf           182.0     1.0
902  45e2642db72311edae74cee2c33601bf           439.0     0.0
819  45e2642db72311edae74cee2c33601bf           182.0     0.0
698  45e2642db72311edae74cee2c33601bf           189.0     0.0
443  45e2642db72311edae74cee2c33601bf           439.0     0.0
952  6672936abb0611edb15ee60116a18e06           753.0     1.0
584  6672936abb0611edb15ee60116a18e06           298.0     0.0
899  7d4cb128b13e11ed9344faffc215a4c7           521.0     0.0
439  7d4cb128b13e11ed9344faffc215a4c7           532.0     1.0
68   86747f24a40a11edbb9e7e0234c0a9a6          1170.0     0.0
850  86747f24a40a11edbb9e7e0234c0a9a6           663.0     1.0
686  86747f24a40a11edbb9e7e0234c0a9a6           956.0     0.0
284  86747f24a40a11edbb9e7e0234c0a9a6           967.0     0.0
124  a47f421ebbb211edb7715225eb050bd9           439.0     1.0
722  a47

In [131]:
duplicate_feedback = comparison_df.duplicated(
    subset=["traveler_id", "destination_id"],
    keep=False
)

print("Duplicate records:", duplicate_feedback.sum())

print(
    comparison_df[duplicate_feedback][
        ["traveler_id", "destination_id", "rating", "accepted"]
    ].sort_values(["traveler_id", "destination_id"])
)

Duplicate records: 182
                          traveler_id  destination_id  rating  accepted
530  02f56846d27e11eda45116bdf5dc994c           178.0     0.0         1
929  02f56846d27e11eda45116bdf5dc994c           178.0     0.0         1
88   07a5f915b93811edbbeba12c98458730            46.0     0.0         1
820  07a5f915b93811edbbeba12c98458730            46.0     0.0         1
324  0c0235fec9d211edb374e63ec1efde2d           956.0     0.0         1
..                                ...             ...     ...       ...
945  fda4d6f5a4c011ed86caab0c117d11ba           600.0     0.0         1
286  fda4d6f5a4c011ed86caab0c117d11ba           788.0     0.0         1
852  fda4d6f5a4c011ed86caab0c117d11ba           788.0     0.0         1
737  ff3fee84d28111edadc00e96bc2eb5a0            81.0     0.0         1
998  ff3fee84d28111edadc00e96bc2eb5a0            81.0     0.0         1

[182 rows x 4 columns]


In [132]:
duplicate_rating_check = (
    comparison_df.groupby(["traveler_id", "destination_id"])["rating"]
    .nunique()
)

print("Total repeated pairs:", duplicate_rating_check.gt(1).sum())
print("Pairs with conflicting ratings:", duplicate_rating_check.eq(2).sum())

Total repeated pairs: 2
Pairs with conflicting ratings: 2


In [133]:
conflicting_pairs = duplicate_rating_check[
    duplicate_rating_check > 1
].index

print(
    comparison_df.set_index(
        ["traveler_id", "destination_id"]
    ).loc[conflicting_pairs][
        ["rating", "accepted"]
    ]
)

                                                 rating  accepted
traveler_id                      destination_id                  
45e2642db72311edae74cee2c33601bf 182.0              1.0         1
                                 182.0              0.0         1
e111955ebb1e11edb15ee60116a18e06 538.0              1.0         1
                                 538.0              0.0         1


In [134]:
conflicting_traveler_ids = [
    "45e2642db72311edae74cee2c33601bf",
    "e111955ebb1e11edb15ee60116a18e06"
]

print(
    original_df[
        original_df["id"].isin(conflicting_traveler_ids)
    ][
        ["id"] +
        [f"Rec_{i}" for i in range(10)] +
        [f"Rating_{i}" for i in range(10)]
    ].to_string(index=False)
)

                              id              Rec_0              Rec_1                                    Rec_2                 Rec_3            Rec_4              Rec_5              Rec_6                                    Rec_7                 Rec_8            Rec_9  Rating_0  Rating_1  Rating_2  Rating_3  Rating_4  Rating_5  Rating_6  Rating_7  Rating_8  Rating_9
45e2642db72311edae74cee2c33601bf Dubrovnik, Croatia     Cannes, France                         La Spezia, Italy Chania, Crete, Greece   Split, Croatia Dubrovnik, Croatia     Cannes, France                         La Spezia, Italy Chania, Crete, Greece   Split, Croatia       1.0       0.0       0.0       0.0       0.0       0.0       0.0       0.0       0.0       0.0
e111955ebb1e11edb15ee60116a18e06      Tulum, Mexico Dubrovnik, Croatia Atlantic City, New Jersey, United States        Monaco, Monaco La Spezia, Italy      Tulum, Mexico Dubrovnik, Croatia Atlantic City, New Jersey, United States        Monaco, Monaco La Spezia,

In [135]:
rec_columns = [f"Rec_{i}" for i in range(10)]

def has_repeated_recommendations(row):
    destinations = [
        str(value).strip()
        for value in row[rec_columns]
        if pd.notna(value) and str(value).strip() != "-1"
    ]
    return len(destinations) != len(set(destinations))

repeated_rows = original_df[
    original_df.apply(has_repeated_recommendations, axis=1)
]

print("Rows with repeated recommendations:", len(repeated_rows))
print("Total original rows:", len(original_df))

Rows with repeated recommendations: 879
Total original rows: 80301


In [136]:
def first_five_equal_last_five(row):
    first = row[[f"Rec_{i}" for i in range(5)]].astype(str).tolist()
    last = row[[f"Rec_{i}" for i in range(5, 10)]].astype(str).tolist()
    return first == last

print(
    "First five equal last five:",
    original_df.apply(first_five_equal_last_five, axis=1).sum()
)

First five equal last five: 67035


In [137]:
rating_columns = [f"Rating_{i}" for i in range(10)]

same_rating_pattern = (
    original_df.apply(
        lambda row: row[
            [f"Rating_{i}" for i in range(5)]
        ].reset_index(drop=True).equals(
            row[
                [f"Rating_{i}" for i in range(5, 10)]
            ].reset_index(drop=True)
        ),
        axis=1
    )
)

print("Rows with identical first-five and last-five ratings:",
      same_rating_pattern.sum())

Rows with identical first-five and last-five ratings: 69076


In [138]:
same_recommendations = original_df.apply(
    first_five_equal_last_five, axis=1
)

both_identical = same_recommendations & same_rating_pattern

print("Both recommendations and ratings identical:",
      both_identical.sum())

Both recommendations and ratings identical: 67028


In [139]:
exceptional_rows = original_df[
    same_recommendations & ~same_rating_pattern
]

print(
    exceptional_rows[
        ["id"] +
        [f"Rec_{i}" for i in range(10)] +
        [f"Rating_{i}" for i in range(10)]
    ].to_string(index=False)
)

                              id                       Rec_0                   Rec_1                                    Rec_2                                    Rec_3                               Rec_4                       Rec_5                   Rec_6                                    Rec_7                                    Rec_8                               Rec_9  Rating_0  Rating_1  Rating_2  Rating_3  Rating_4  Rating_5  Rating_6  Rating_7  Rating_8  Rating_9
fb10169ed28211eda45116bdf5dc994c Dubai, United Arab Emirates            Nice, France                   Cabo San Lucas, Mexico Lake Buena Vista, Florida, United States    Key West, Florida, United States Dubai, United Arab Emirates            Nice, France                   Cabo San Lucas, Mexico Lake Buena Vista, Florida, United States    Key West, Florida, United States       0.0       0.0       0.0       0.0       0.0       1.0       1.0       0.0       0.0       0.0
7a92eda0a69211edbc152eed40b3d78f         Ljubljana, Sl

In [140]:
for i in range(10):
    col = f"Rating_{i}"

    valid_count = original_df[col].isin([0, 1]).sum()

    print(f"{col}: {valid_count}")

Rating_0: 12929
Rating_1: 13173
Rating_2: 13211
Rating_3: 13218
Rating_4: 13231
Rating_5: 2431
Rating_6: 2560
Rating_7: 2556
Rating_8: 2557
Rating_9: 2551


In [141]:
print("Total destinations:", len(destination_names))

print("First 15 destinations:")
for destination in destination_names[:15]:
    print(destination)

Total destinations: 1486
First 15 destinations:
Butrint, Albania
Tirana, Albania
Algiers, Algeria
Oran, Algeria
Andorra la Vella, Andorra
Jolly Harbour, Antigua and Barbuda
St John's, Antigua and Barbuda
Buenos Aires, Argentina
Córdoba, Argentina
Mar del Plata, Argentina
Mendoza, Argentina
Puerto Iguazu, Argentina
Puerto Madryn, Argentina
Salta, Argentina
San Carlos de Bariloche, Argentina


In [142]:
destination_df = pd.DataFrame({
    "destination_id": range(len(destination_names)),
    "destination_name": destination_names
})

destination_df["country"] = (
    destination_df["destination_name"]
    .str.split(", ")
    .str[-1]
)

print(destination_df.head(15))

    destination_id                    destination_name              country
0                0                    Butrint, Albania              Albania
1                1                     Tirana, Albania              Albania
2                2                    Algiers, Algeria              Algeria
3                3                       Oran, Algeria              Algeria
4                4           Andorra la Vella, Andorra              Andorra
5                5  Jolly Harbour, Antigua and Barbuda  Antigua and Barbuda
6                6      St John's, Antigua and Barbuda  Antigua and Barbuda
7                7             Buenos Aires, Argentina            Argentina
8                8                  Córdoba, Argentina            Argentina
9                9            Mar del Plata, Argentina            Argentina
10              10                  Mendoza, Argentina            Argentina
11              11            Puerto Iguazu, Argentina            Argentina
12          

In [143]:
country_counts = (
    destination_df["country"]
    .value_counts()
    .reset_index()
)

country_counts.columns = ["country", "destination_count"]

print("Total countries:", destination_df["country"].nunique())
print(country_counts.head(15))

Total countries: 143
           country  destination_count
0    United States                603
1            Italy                 36
2           France                 35
3          Germany                 34
4            Spain                 33
5   United Kingdom                 26
6           Greece                 25
7           Mexico                 21
8           Brazil                 21
9            China                 19
10           Egypt                 17
11          Canada                 16
12     Switzerland                 16
13         Austria                 15
14       Australia                 15


In [144]:
destination_df["city_or_region"] = (
    destination_df["destination_name"]
    .str.rsplit(", ", n=1)
    .str[0]
)

print(
    destination_df[
        ["destination_id", "city_or_region", "country"]
    ].head(15)
)

    destination_id           city_or_region              country
0                0                  Butrint              Albania
1                1                   Tirana              Albania
2                2                  Algiers              Algeria
3                3                     Oran              Algeria
4                4         Andorra la Vella              Andorra
5                5            Jolly Harbour  Antigua and Barbuda
6                6                St John's  Antigua and Barbuda
7                7             Buenos Aires            Argentina
8                8                  Córdoba            Argentina
9                9            Mar del Plata            Argentina
10              10                  Mendoza            Argentina
11              11            Puerto Iguazu            Argentina
12              12            Puerto Madryn            Argentina
13              13                    Salta            Argentina
14              14  San C

In [145]:
print(
    "Total destination records:",
    len(destination_df)
)

print(
    "Unique destination names:",
    destination_df["destination_name"].nunique()
)

print(
    "Duplicate destination names:",
    destination_df["destination_name"].duplicated().sum()
)

Total destination records: 1486
Unique destination names: 1486
Duplicate destination names: 0


In [146]:
print(original_df.columns.tolist())

['id', 'form_a', 'form_b', 'form_c', 'form_f', 'form_g', 'form_h', 'form_i', 'form_j', 'form_k', 'form_r', 'form_rr', 'yes_swipes', 'no_swipes', 'maybe_swipes', 'Model', 'Retrieval', 'DynaMatch', 'Rating_0', 'Rating_1', 'Rating_2', 'Rating_3', 'Rating_4', 'Rating_5', 'Rating_6', 'Rating_7', 'Rating_8', 'Rating_9', 'Rec_0', 'Rec_1', 'Rec_2', 'Rec_3', 'Rec_4', 'Rec_5', 'Rec_6', 'Rec_7', 'Rec_8', 'Rec_9']


In [147]:
print(destination_df.info())
print(destination_df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1486 entries, 0 to 1485
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   destination_id    1486 non-null   int64 
 1   destination_name  1486 non-null   object
 2   country           1486 non-null   object
 3   city_or_region    1486 non-null   object
dtypes: int64(1), object(3)
memory usage: 46.6+ KB
None
destination_id      0
destination_name    0
country             0
city_or_region      0
dtype: int64


In [148]:
places_df = pd.read_csv("data/external/places.csv")

print("Dataset shape:", places_df.shape)

display(places_df.head())

Dataset shape: (33289, 33)


,wikidata_id,name_en,name_fr,name_de,name_es,name_pt,name_it,name_hi,name_zh,name_cs,...,url_de,url_es,url_pt,url_it,url_hi,url_zh,url_cs,url_ja,url_ru,search_impressions_decile
0,Q1034660,Capital Gate,Capital Gate,Capital Gate,Capital Gate,Capital Gate,Capital Gate,कैपिटल गेट,首都門,Capital Gate,...,https://audiala.com/de/vereinigte-arabische-em...,https://audiala.com/es/emiratos-arabes-unidos/...,https://audiala.com/pt/emirados-arabes-unidos/...,https://audiala.com/it/emirati-arabi-uniti/abu...,https://audiala.com/hi/संयुक्त-अरब-अमीरात/अबू-...,https://audiala.com/zh/阿拉伯联合酋长国/阿布扎比/首都門,https://audiala.com/cs/spojene-arabske-emiraty...,https://audiala.com/ja/アラブ首長国連邦/アブダビ/キャピタルゲートビル,https://audiala.com/ru/объединённые-арабские-э...,7
1,Q1321116,Guggenheim Abu Dhabi,Guggenheim Abou Dabi,Guggenheim Abu Dhabi,Guggenheim Abu Dhabi,Guggenheim Abu Dhabi,Guggenheim Abu Dhabi,गुग्गेनहाइम अबू धाबी,阿布達比古根漢博物館,Guggenheim Abu Dhabi,...,https://audiala.com/de/vereinigte-arabische-em...,https://audiala.com/es/emiratos-arabes-unidos/...,https://audiala.com/pt/emirados-arabes-unidos/...,https://audiala.com/it/emirati-arabi-uniti/abu...,https://audiala.com/hi/संयुक्त-अरब-अमीरात/अबू-...,https://audiala.com/zh/阿拉伯联合酋长国/阿布扎比/阿布達比古根漢博物館,https://audiala.com/cs/spojene-arabske-emiraty...,https://audiala.com/ja/アラブ首長国連邦/アブダビ/グッゲンハイム-ア...,https://audiala.com/ru/объединённые-арабские-э...,10
2,Q134392411,Disneyland Abu Dhabi,Disneyland Abu Dhabi,Disneyland Abu Dhabi,Disneyland Abu Dhabi,Disneyland Abu Dhabi,Disneyland Abu Dhabi,डिज़नीलैंड अबू धाबी,阿布扎比迪士尼乐园,Disneyland Abu Dhabi,...,https://audiala.com/de/vereinigte-arabische-em...,https://audiala.com/es/emiratos-arabes-unidos/...,https://audiala.com/pt/emirados-arabes-unidos/...,https://audiala.com/it/emirati-arabi-uniti/abu...,https://audiala.com/hi/संयुक्त-अरब-अमीरात/अबू-...,https://audiala.com/zh/阿拉伯联合酋长国/阿布扎比/阿布扎比迪士尼乐园,https://audiala.com/cs/spojene-arabske-emiraty...,https://audiala.com/ja/アラブ首長国連邦/アブダビ/ディズニーランド-...,https://audiala.com/ru/объединённые-арабские-э...,10
3,Q1371570,Etihad Towers,Etihad Towers,Etihad Towers,Etihad Towers,Etihad Towers,Etihad Towers,एतिहाद टावर्स,阿提哈德塔,Etihad Towers,...,https://audiala.com/de/vereinigte-arabische-em...,https://audiala.com/es/emiratos-arabes-unidos/...,https://audiala.com/pt/emirados-arabes-unidos/...,https://audiala.com/it/emirati-arabi-uniti/abu...,https://audiala.com/hi/संयुक्त-अरब-अमीरात/अबू-...,https://audiala.com/zh/阿拉伯联合酋长国/阿布扎比/阿提哈德塔,https://audiala.com/cs/spojene-arabske-emiraty...,https://audiala.com/ja/アラブ首長国連邦/アブダビ/エティハドタワー,https://audiala.com/ru/объединённые-арабские-э...,3
4,Q1435741,Sky Tower,Sky Tower,Sky Tower,Sky Tower,Sky Tower Dubai,Sky Tower (Abu Dhabi),"स्काई टावर, अबु धाबी",天空塔,Sky Tower,...,https://audiala.com/de/vereinigte-arabische-em...,https://audiala.com/es/emiratos-arabes-unidos/...,https://audiala.com/pt/emirados-arabes-unidos/...,https://audiala.com/it/emirati-arabi-uniti/abu...,https://audiala.com/hi/संयुक्त-अरब-अमीरात/अबू-...,https://audiala.com/zh/阿拉伯联合酋长国/阿布扎比/天空塔,https://audiala.com/cs/spojene-arabske-emiraty...,https://audiala.com/ja/アラブ首長国連邦/アブダビ/スカイタワー-アブダビ,https://audiala.com/ru/объединённые-арабские-э...,9


In [149]:
for column in places_df.columns:
    print(column)
    

wikidata_id
name_en
name_fr
name_de
name_es
name_pt
name_it
name_hi
name_zh
name_cs
name_ja
name_ru
latitude
longitude
country_iso2
country_en
city_en
category
article_tier
wikidata_pagerank
sitelinks
url_en
url_fr
url_de
url_es
url_pt
url_it
url_hi
url_zh
url_cs
url_ja
url_ru
search_impressions_decile


In [150]:
useful_columns = [
    "wikidata_id",
    "name_en",
    "latitude",
    "longitude",
    "country_en",
    "city_en",
    "category",
    "article_tier",
    "wikidata_pagerank"
]

places_selected = places_df[useful_columns].copy()

print("Selected dataset shape:", places_selected.shape)

display(places_selected.head())

Selected dataset shape: (33289, 9)


,wikidata_id,name_en,latitude,longitude,country_en,city_en,category,article_tier,wikidata_pagerank
0,Q1034660,Capital Gate,24.418636,54.434692,United Arab Emirates,Abu Dhabi,building,legacy,1.446364
1,Q1321116,Guggenheim Abu Dhabi,24.539514,54.398958,United Arab Emirates,Abu Dhabi,attraction,legacy,2.252466
2,Q134392411,Disneyland Abu Dhabi,24.485853,54.602978,United Arab Emirates,Abu Dhabi,amusement-park,legacy,NaN
3,Q1371570,Etihad Towers,24.459322,54.321297,United Arab Emirates,Abu Dhabi,tower,legacy,4.051022
4,Q1435741,Sky Tower,24.495948,54.407514,United Arab Emirates,Abu Dhabi,building,legacy,1.461286


In [151]:
print("Unique categories:",
      places_selected["category"].nunique())

print("\nTop 20 attraction categories:")

print(
    places_selected["category"]
    .value_counts()
    .head(20)
)

Unique categories: 50

Top 20 attraction categories:
category
building               5477
museum                 4428
attraction             3020
statue                 2218
church                 2082
monument               1525
theatre                1381
palace                 1142
square                 1066
fortification           914
stadium                 839
park                    834
bridge                  754
archaeological-site     678
temple                  524
locality                514
garden                  481
mosque                  461
cathedral               457
library                 426
Name: count, dtype: int64


In [152]:
print("Total rows:", len(places_selected))

print("\nMissing values:")
print(places_selected.isnull().sum())

print("\nDuplicate Wikidata IDs:",
      places_selected["wikidata_id"].duplicated().sum())

print("\nDuplicate attraction names:",
      places_selected["name_en"].duplicated().sum())

Total rows: 33289

Missing values:
wikidata_id             0
name_en                25
latitude                0
longitude               0
country_en              0
city_en                 0
category                0
article_tier            0
wikidata_pagerank    5579
dtype: int64

Duplicate Wikidata IDs: 0

Duplicate attraction names: 600


In [153]:
places_selected = places_selected.dropna(subset=["name_en"]).copy()

print("Rows after removing missing names:", len(places_selected))
print("Missing attraction names:", places_selected["name_en"].isnull().sum())

Rows after removing missing names: 33264
Missing attraction names: 0


In [154]:
print("Destination names:")
print(destination_df["destination_name"].head(10).to_list())

print("\nAttraction locations:")
print(places_selected[["name_en", "city_en", "country_en"]].head(10))

Destination names:
['Butrint, Albania', 'Tirana, Albania', 'Algiers, Algeria', 'Oran, Algeria', 'Andorra la Vella, Andorra', 'Jolly Harbour, Antigua and Barbuda', "St John's, Antigua and Barbuda", 'Buenos Aires, Argentina', 'Córdoba, Argentina', 'Mar del Plata, Argentina']

Attraction locations:
                     name_en    city_en            country_en
0               Capital Gate  Abu Dhabi  United Arab Emirates
1       Guggenheim Abu Dhabi  Abu Dhabi  United Arab Emirates
2       Disneyland Abu Dhabi  Abu Dhabi  United Arab Emirates
3              Etihad Towers  Abu Dhabi  United Arab Emirates
4                  Sky Tower  Abu Dhabi  United Arab Emirates
5  Zayed Sports City Stadium  Abu Dhabi  United Arab Emirates
6   Hili Archaeological Park  Abu Dhabi  United Arab Emirates
7               The Landmark  Abu Dhabi  United Arab Emirates
8      Sowwah Square Tower 2  Abu Dhabi  United Arab Emirates
9        Sheikh Zayed Bridge  Abu Dhabi  United Arab Emirates


In [155]:
# Create a standardized matching key in destination data
destination_df["match_key"] = (
    destination_df["city_or_region"].str.split(",").str[0].str.strip().str.lower()
    + "|" +
    destination_df["country"].str.strip().str.lower()
)

# Create the same key in attraction data
places_selected["match_key"] = (
    places_selected["city_en"].str.strip().str.lower()
    + "|" +
    places_selected["country_en"].str.strip().str.lower()
)

# Match attractions with destinations
matched_places = places_selected.merge(
    destination_df[["destination_id", "destination_name", "match_key"]],
    on="match_key",
    how="inner"
)

print("Total destinations:", len(destination_df))
print("Matched attraction records:", len(matched_places))
print("Destinations with attractions:", matched_places["destination_id"].nunique())
print("\nSample matches:")
print(matched_places[["destination_name", "name_en", "category"]].head(10))

Total destinations: 1486
Matched attraction records: 24243
Destinations with attractions: 307

Sample matches:
                  destination_name                    name_en  \
0  Abu Dhabi, United Arab Emirates               Capital Gate   
1  Abu Dhabi, United Arab Emirates       Guggenheim Abu Dhabi   
2  Abu Dhabi, United Arab Emirates       Disneyland Abu Dhabi   
3  Abu Dhabi, United Arab Emirates              Etihad Towers   
4  Abu Dhabi, United Arab Emirates                  Sky Tower   
5  Abu Dhabi, United Arab Emirates  Zayed Sports City Stadium   
6  Abu Dhabi, United Arab Emirates   Hili Archaeological Park   
7  Abu Dhabi, United Arab Emirates               The Landmark   
8  Abu Dhabi, United Arab Emirates      Sowwah Square Tower 2   
9  Abu Dhabi, United Arab Emirates        Sheikh Zayed Bridge   

              category  
0             building  
1           attraction  
2       amusement-park  
3                tower  
4             building  
5              stadium 

In [156]:
# Count attractions in each destination category
category_features = pd.crosstab(
    matched_places["destination_id"],
    matched_places["category"]
)

# Add attraction count
category_features["total_attractions"] = category_features.sum(axis=1)

# Join features with all destinations
destination_features = destination_df.set_index("destination_id").join(
    category_features,
    how="left"
).fillna(0)

print("Destination feature shape:", destination_features.shape)
print("\nSample destination features:")
print(destination_features[["destination_name", "total_attractions"]].head(10))

Destination feature shape: (1486, 55)

Sample destination features:
                                  destination_name  total_attractions
destination_id                                                       
0                                 Butrint, Albania                0.0
1                                  Tirana, Albania               45.0
2                                 Algiers, Algeria                0.0
3                                    Oran, Algeria                8.0
4                        Andorra la Vella, Andorra                0.0
5               Jolly Harbour, Antigua and Barbuda                0.0
6                   St John's, Antigua and Barbuda                0.0
7                          Buenos Aires, Argentina              269.0
8                               Córdoba, Argentina                2.0
9                         Mar del Plata, Argentina                0.0


In [157]:
# Group attraction categories into travel interests
interest_mapping = {
    "history": [
        "monument", "archaeological-site", "fortification",
        "castle", "memorial", "ruins"
    ],
    "culture": [
        "museum", "theatre", "artwork", "gallery",
        "library", "opera-house"
    ],
    "religious": [
        "church", "temple", "mosque", "cathedral",
        "monastery", "shrine"
    ],
    "nature": [
        "park", "garden", "beach", "waterfall",
        "mountain", "lake", "island"
    ],
    "architecture": [
        "building", "bridge", "tower", "palace",
        "square"
    ],
    "entertainment": [
        "stadium", "amusement-park", "zoo",
        "aquarium", "theme-park"
    ]
}

# Create interest scores
for interest, categories in interest_mapping.items():
    available_categories = [
        category for category in categories
        if category in destination_features.columns
    ]

    destination_features[interest] = (
        destination_features[available_categories].sum(axis=1)
    )

print(destination_features[
    ["destination_name", "history", "culture",
     "religious", "nature", "architecture", "entertainment"]
].head(10))

                                  destination_name  history  culture  \
destination_id                                                         
0                                 Butrint, Albania      0.0      0.0   
1                                  Tirana, Albania     10.0      6.0   
2                                 Algiers, Algeria      0.0      0.0   
3                                    Oran, Algeria      1.0      0.0   
4                        Andorra la Vella, Andorra      0.0      0.0   
5               Jolly Harbour, Antigua and Barbuda      0.0      0.0   
6                   St John's, Antigua and Barbuda      0.0      0.0   
7                          Buenos Aires, Argentina     15.0    111.0   
8                               Córdoba, Argentina      0.0      0.0   
9                         Mar del Plata, Argentina      0.0      0.0   

                religious  nature  architecture  entertainment  
destination_id                                                  
0    

In [158]:
from sklearn.preprocessing import MinMaxScaler

interest_columns = [
    "history", "culture", "religious",
    "nature", "architecture", "entertainment"
]

scaler = MinMaxScaler()

destination_features[interest_columns] = scaler.fit_transform(
    destination_features[interest_columns]
)

print(destination_features[
    ["destination_name"] + interest_columns
].head(10))

                                  destination_name   history   culture  \
destination_id                                                           
0                                 Butrint, Albania  0.000000  0.000000   
1                                  Tirana, Albania  0.093458  0.026316   
2                                 Algiers, Algeria  0.000000  0.000000   
3                                    Oran, Algeria  0.009346  0.000000   
4                        Andorra la Vella, Andorra  0.000000  0.000000   
5               Jolly Harbour, Antigua and Barbuda  0.000000  0.000000   
6                   St John's, Antigua and Barbuda  0.000000  0.000000   
7                          Buenos Aires, Argentina  0.140187  0.486842   
8                               Córdoba, Argentina  0.000000  0.000000   
9                         Mar del Plata, Argentina  0.000000  0.000000   

                religious    nature  architecture  entertainment  
destination_id                              

In [159]:
def recommend_places(user_interests, top_n=5):

    # Select destinations with attraction information
    available_places = destination_features[
        destination_features["total_attractions"] > 0
    ].copy()

    # Calculate average preference match
    available_places["match_score"] = (
        available_places[user_interests].mean(axis=1)
    )

    # Sort destinations by match score
    recommendations = available_places.sort_values(
        by="match_score",
        ascending=False
    )

    return recommendations[
        ["destination_name", "match_score", "total_attractions"]
    ].head(top_n)

In [160]:
# Save destination features for the Streamlit application
destination_features.to_csv(
    "data/destination_features.csv",
    index=True
)

print("Destination features saved successfully!")
print("Total destinations:", len(destination_features))

Destination features saved successfully!
Total destinations: 1486


In [161]:
matched_places[
    ["destination_id", "destination_name", "name_en", "category"]
].to_csv(
    "data/matched_attractions.csv",
    index=False
)

print("Saved:", len(matched_places), "attraction records")

Saved: 24243 attraction records
